# COSMOS ZoomInOut — v5: Veri Kümesi Kalitesi Düzeltmeleri

v4'te bulduğumuz iki kök-neden düzeltiliyor:
1. **`shape_feat`/`texture_feat` doygun** (özellikle shape, medyan cosine-sim
   1.000) -- zenginleştirilmiş tanımlayıcılar (LBP doku, Hu momentleri şekil)
   + modlar-arası kalibre edilmiş `hardness` formülü ile düzeltiliyor.
2. **Temel görsel-metin eşleşmesi doğrulanmamış** (COCO görseli + alakasız
   Türkçe hikaye korpusu, ~0.62 ortalama benzerlik -- rastgele taban çizgisinden
   [~0.38] güçlü ayrışıyor ama "hassas caption" garantisi yok) -- COCO'nun
   kendi orijinal caption'ları bulunabilirse, pozitif çift tanımı buna
   geçiriliyor.

**Zincire (traversal, txt_shift) DOKUNULMADI** -- sadece hard-negative mining
(görsel CV özellikleri) ve ana hizalama çiftlerinin (train_pairs/test_pairs)
kaynağı değişiyor.

**Bölüm 1'i (COCO teşhisi) İLK çalıştır ve çıktısını oku** -- geri kalan
akışın COCO caption'ı kullanıp kullanmayacağını o belirliyor. Dosya
bulunamazsa/eşleşme kurulamazsa notebook otomatik olarak v4'ün mevcut
davranışına (en-yakın-hikaye-cümlesi) düşer, çalışmaya devam eder.


## 1. Kurulum

In [ ]:
# HÜCRE 1: Kurulum ve içe aktarımlar
!pip install -q datasets opencv-python Pillow tqdm nltk scikit-image

import os
import re
import copy
import pickle
import random
import secrets
import itertools
import textwrap
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Set

import cv2
import nltk
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from tqdm.auto import tqdm

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Hazır! Aktif cihaz: {device}")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DRIVE_BASE = "/content/drive/MyDrive/COSMOS"

# --- v5'e ÖZEL dosya adları -- v4/Fikir2 ile ÇAKIŞMAZ, paralel çalıştırılabilir.
# CV_CACHE_PATH kasıtlı olarak YENİ bir isim -- ozellikler zenginlestirildigi
# icin v4'un eski onbellegini KULLANMIYORUZ (yoksa eski, zayif ozellikler
# sessizce geri yuklenir).
TRAIN_PAIRS_PATH   = f"{DRIVE_BASE}/ZoomInOut/v5i_train_pairs.pkl"  # GERI DONULDU: v5i (v5j denemesi ayri tutuluyor, silinmedi)
TEST_PAIRS_PATH    = f"{DRIVE_BASE}/ZoomInOut/v5i_test_pairs.pkl"  # GERI DONULDU: v5i
VAL_PAIRS_PATH     = f"{DRIVE_BASE}/ZoomInOut/v5i_val_pairs.pkl"  # GERI DONULDU: v5i
CHAIN_RECORDS_PATH = f"{DRIVE_BASE}/ZoomInOut/v5i_chain_records.pkl"  # GERI DONULDU: v5i
CV_CACHE_PATH      = f"{DRIVE_BASE}/ZoomInOut/v5_cv_feature_cache_enriched.pkl"
MODE_CALIB_PATH    = f"{DRIVE_BASE}/ZoomInOut/v5_mode_calibration_percentile.pkl"  # v5b: eski (mean,std) onbellegiyle CAKISMASIN diye yeni isim
COCO_CAPTIONS_EMB_PATH = f"{DRIVE_BASE}/ZoomInOut/v5_coco_captions_embedded.pkl"


## 2. Veri Sınıfları ve İndeks (v4 ile aynı)

In [ ]:
# HÜCRE 2: Dataclass'lar

@dataclass
class ImagePatchMeta:
    image_id:         str
    patch_id:         str
    bbox:             list
    embedding:        np.ndarray
    objectness_score: float = 1.0
    type:             str   = "object"
    window_type:      str   = ""
    area:             float = 0.0
    attention_score:  float = 0.0
    color_feat:       Optional[np.ndarray] = None
    shape_feat:       Optional[np.ndarray] = None
    texture_feat:     Optional[np.ndarray] = None

    def iou(self, other: "ImagePatchMeta") -> float:
        x1 = max(self.bbox[0], other.bbox[0])
        y1 = max(self.bbox[1], other.bbox[1])
        x2 = min(self.bbox[2], other.bbox[2])
        y2 = min(self.bbox[3], other.bbox[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        if inter == 0:
            return 0.0
        a1 = (self.bbox[2] - self.bbox[0]) * (self.bbox[3] - self.bbox[1])
        a2 = (other.bbox[2] - other.bbox[0]) * (other.bbox[3] - other.bbox[1])
        return inter / (a1 + a2 - inter + 1e-10)


@dataclass
class TextPatchMeta:
    doc_id:         str
    patch_id:       str
    text_content:   str
    embedding:      np.ndarray
    win_size:       int   = 2
    win_label:      str   = "w2"
    richness_score: float = 0.0
    sent_range:     tuple = field(default_factory=lambda: (0, 0))
    lexical_feat:   Optional[np.ndarray] = None  # TF-IDF -- metin-tarafı hard-negative madenciliği için


@dataclass
class HybridStepRecord:
    step:            int
    action:          str
    modality:        str
    image_id:        Optional[str]   = None
    patch_id:        Optional[str]   = None
    bbox:            Optional[list]  = None
    doc_id:          Optional[str]   = None
    text_content:    Optional[str]   = None
    similarity:      float           = 0.0
    similarity_mode: str             = "semantic"
    verified_label:  str             = "positive"
    # Metin-tarafı hard-negative (sadece img_to_txt/txt_to_img adımlarında dolu olur):
    hard_negative_txt_patch_id: Optional[str]   = None
    hard_negative_txt_sim:      Optional[float] = None
    hard_negative_txt_mode:     Optional[str]   = None


In [ ]:
# HÜCRE 3: HybridMultimodalIndex

class HybridMultimodalIndex:
    def __init__(self):
        self.image_patches: List[ImagePatchMeta] = []
        self.text_patches:  List[TextPatchMeta]  = []
        self._img_matrix:   Optional[np.ndarray] = None
        self._txt_matrix:   Optional[np.ndarray] = None
        self._lex_matrix:   Optional[np.ndarray] = None
        self._img_dirty = True
        self._txt_dirty = True
        self._lex_dirty = True

    def _is_quality_text(self, text: str) -> bool:
        if not text or len(text.split()) < 5:
            return False
        if text.strip().endswith("?"):
            return False
        BAGLAC = {"ve", "ama", "fakat", "lakin", "çünkü", "ancak",
                  "ile", "ya", "da", "de", "ki", "bu", "şu", "o",
                  "diye", "hem", "ne", "ah", "oh", "hmm"}
        ilk = text.split()[0].lower().rstrip(".,!?")
        if ilk in BAGLAC:
            return False
        if text.startswith('"') or text.startswith('-') or '—' in text:
            return False
        return True

    def add_image_patch(self, patch: ImagePatchMeta):
        for attr in ("embedding", "color_feat", "shape_feat", "texture_feat"):
            feat = getattr(patch, attr, None)
            if feat is not None:
                n = np.linalg.norm(feat)
                if n > 0:
                    setattr(patch, attr, feat / n)
        self.image_patches.append(patch)
        self._img_dirty = True

    def add_text_patch(self, patch: TextPatchMeta):
        n = np.linalg.norm(patch.embedding)
        if n > 0:
            patch.embedding = patch.embedding / n
        if patch.lexical_feat is not None:
            ln = np.linalg.norm(patch.lexical_feat)
            if ln > 0:
                patch.lexical_feat = patch.lexical_feat / ln
        self.text_patches.append(patch)
        self._txt_dirty = True
        self._lex_dirty = True

    def _build_matrices(self):
        if self._img_dirty and self.image_patches:
            self._img_matrix = np.stack([p.embedding for p in self.image_patches], axis=0).astype(np.float32)
            self._img_dirty = False
        if self._txt_dirty and self.text_patches:
            self._txt_matrix = np.stack([p.embedding for p in self.text_patches], axis=0).astype(np.float32)
            self._txt_dirty = False
        if self._lex_dirty and self.text_patches and all(p.lexical_feat is not None for p in self.text_patches):
            self._lex_matrix = np.stack([p.lexical_feat for p in self.text_patches], axis=0).astype(np.float32)
            self._lex_dirty = False

    def query_text(self, query_emb, exclude_doc_ids=None, threshold=-1.0, min_win_size=1, top_k=1):
        """v5i: top_k>1 verilirse, tek-en-iyi yerine EN IYI top_k adayi
        (agirlikli skora gore sirali) dondurur -- motor bunlar arasindan
        cesitlilik-farkinda secim yapabilsin diye (bkz. MultimodalSpiralEngine)."""
        self._build_matrices()
        if self._txt_matrix is None or not self.text_patches:
            return []
        exclude_doc_ids = exclude_doc_ids or set()
        q = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        raw = np.clip(self._txt_matrix @ q, -1.0, 1.0)
        WIN_WEIGHT = {1: 0.70, 2: 1.05, 3: 1.15}
        weighted = raw.copy()
        for i, p in enumerate(self.text_patches):
            if p.doc_id in exclude_doc_ids or p.win_size < min_win_size or not self._is_quality_text(p.text_content):
                weighted[i] = -999.0
            else:
                weighted[i] = weighted[i] * WIN_WEIGHT.get(p.win_size, 1.0)
        if top_k <= 1:
            best_idx = int(np.argmax(weighted))
            if weighted[best_idx] <= -999.0:
                return []
            return [(self.text_patches[best_idx], float(raw[best_idx]))]
        order = np.argsort(-weighted)[:top_k]
        return [(self.text_patches[i], float(raw[i])) for i in order if weighted[i] > -999.0]

    def query_image(self, query_emb, exclude_image_ids=None, top_k=1):
        """v5i: top_k>1 verilirse, tek-en-iyi yerine EN IYI top_k adayi dondurur."""
        self._build_matrices()
        if self._img_matrix is None or not self.image_patches:
            return []
        exclude_image_ids = exclude_image_ids or set()
        q = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        raw = np.clip(self._img_matrix @ q, -1.0, 1.0)
        TYPE_WEIGHT = {"object": 1.10, "background": 0.92}
        weighted = raw.copy()
        for i, p in enumerate(self.image_patches):
            if p.image_id in exclude_image_ids:
                weighted[i] = -999.0
            else:
                weighted[i] = weighted[i] * TYPE_WEIGHT.get(p.type, 1.0)
        if top_k <= 1:
            best_idx = int(np.argmax(weighted))
            if weighted[best_idx] <= -999.0:
                return []
            return [(self.image_patches[best_idx], float(raw[best_idx]))]
        order = np.argsort(-weighted)[:top_k]
        return [(self.image_patches[i], float(raw[i])) for i in order if weighted[i] > -999.0]

    def text_patches_of(self, doc_id):
        return [p for p in self.text_patches if p.doc_id == doc_id]

    def image_patches_of(self, image_id):
        return [p for p in self.image_patches if p.image_id == image_id]

    def image_ids(self):
        # v5h: SIRALI donduruluyor -- set() sirasi Python surecleri arasinda
        # hash-randomizasyonu yuzunden DEGISEBILIR, bu da train/test
        # bolunmesinin oturumdan oturuma FARKLI cikmasina (ve sizintiya)
        # yol aciyordu.
        return sorted({p.image_id for p in self.image_patches})

    def find_hard_negative_text(self, anchor_patch: "TextPatchMeta", exclude_doc_ids,
                                 hardness_min: float = 0.05, hard_neg_sem_thresh: float = 0.85):
        """Fikir 2: metin-tarafı hard-negative arama. anchor_patch ile YÜZEYDE
        (TF-IDF/leksik) benzer ama SEMANTİK olarak farklı, BAŞKA bir dokümandan
        bir metin adayı arar -- görsel tarafındaki renk/şekil/doku mantığının
        metin karşılığı. Traversal'ı ETKİLEMEZ, sadece paralel bir alan olarak
        (img_to_txt adımına) eklenir; txt_shift'in (chain tutarlılığı için
        kritik) semantik-en-yakın seçim mantığına dokunmaz.
        """
        self._build_matrices()
        if self._lex_matrix is None or anchor_patch.lexical_feat is None:
            return None
        surface_all = np.clip(self._lex_matrix @ anchor_patch.lexical_feat, -1.0, 1.0)
        sem_all = np.clip(self._txt_matrix @ anchor_patch.embedding, -1.0, 1.0)
        hardness_all = surface_all - sem_all

        best_idx, best_hardness = -1, -999.0
        for i, p in enumerate(self.text_patches):
            if p.doc_id in exclude_doc_ids or p.doc_id == anchor_patch.doc_id:
                continue
            if not self._is_quality_text(p.text_content):
                continue
            if hardness_all[i] > best_hardness:
                best_idx, best_hardness = i, hardness_all[i]

        if best_idx == -1 or best_hardness < hardness_min or sem_all[best_idx] >= hard_neg_sem_thresh:
            return None
        cand = self.text_patches[best_idx]
        return cand.patch_id, float(surface_all[best_idx]), "lexical"


## 3. Ham Veri Kümelerini Yükle (COCO teşhisinden ÖNCE gerekli)

Bu hücre sadece görsel/metin veri kümelerini yükleyip indekse ekliyor -- CV özellik hesaplaması (yavaş kısım) HENÜZ yok, o Bölüm 3'te.

In [ ]:
# HÜCRE 5a: Ham veri yükleme + indeks kurulumu (CV özellikleri HENÜZ yok)

hybrid_index = HybridMultimodalIndex()

text_dataset_path  = f'{DRIVE_BASE}/ds_sampled_text_datas_tr'
image_dataset_path = f'{DRIVE_BASE}/ds_sampled_image_datas_WH_224'

print("⏳ 1/4: Görsel veri kümesi yükleniyor...")
ds_sampled_image = load_from_disk(image_dataset_path)
print("⏳ 2/4: Metin veri kümesi yükleniyor...")
ds_sampled_text = load_from_disk(text_dataset_path)

print("⏳ 3/4: Metin embedding'leri indekse ekleniyor...")
with open(f'{DRIVE_BASE}/ZoomInOut/simple_story_sentences_embeddings_new_tr_5000.pkl', 'rb') as f:
    all_text_metadata = pickle.load(f)

from sklearn.feature_extraction.text import TfidfVectorizer
_tfidf = TfidfVectorizer(max_features=512, min_df=2)
_lex_matrix = _tfidf.fit_transform([m["text_content"] for m in all_text_metadata]).toarray().astype(np.float32)

for i, txt_m in enumerate(all_text_metadata):
    hybrid_index.add_text_patch(TextPatchMeta(
        doc_id=txt_m["doc_id"], patch_id=txt_m["patch_id"],
        text_content=txt_m["text_content"], embedding=txt_m["embedding"],
        win_size=int(txt_m.get("win_size", 2)), win_label=txt_m.get("win_label", "w2"),
        richness_score=float(txt_m.get("richness_score", 0.0)),
        lexical_feat=_lex_matrix[i],
    ))

print("⏳ 4/4: Görsel embedding'leri indekse ekleniyor...")
with open(f'{DRIVE_BASE}/ZoomInOut/coco_image_dynamic_embeddings_new_WH_224_2048.pkl', 'rb') as f:
    all_image_metadata = pickle.load(f)
for img_m in all_image_metadata:
    hybrid_index.add_image_patch(ImagePatchMeta(
        image_id=img_m["image_id"], patch_id=img_m["patch_id"], bbox=img_m["bbox"],
        embedding=img_m["embedding"], objectness_score=img_m.get("objectness_score", 1.0),
        type=img_m.get("type", "object"),
    ))

print(f"✅ İndeks hazır: {len(hybrid_index.image_patches)} görsel yama, {len(hybrid_index.text_patches)} metin yaması.")


## 4. COCO ID + Caption Teşhisi (ÖNCE BUNU ÇALIŞTIR)

Bu bölüm hiçbir şeyi değiştirmez, sadece bilgi toplar. Amaç: (a) görsel
veri kümesinde orijinal COCO image_id'sinin saklanıp saklanmadığını
görmek, (b) Drive'da bir COCO caption dosyası bulmak, (c) ikisini
eşleştirip kapsamı ölçmek.

In [ ]:
# HÜCRE T1: ds_sampled_image şema kontrolü
print("=== Görsel veri kümesi (ds_sampled_image) şema kontrolü ===")
print("Sütunlar:", ds_sampled_image.column_names)
print("\nİlk kayıt (piksel/embedding hariç, tüm alanlar):")
first = ds_sampled_image[0]
for k, v in first.items():
    if k not in ("image",):
        print(f"  {k}: {repr(v)[:200]}")

candidate_id_fields = [k for k in ds_sampled_image.column_names
                        if any(s in k.lower() for s in ["id", "file", "name", "coco", "path"])]
print("\nOlası COCO-ID/dosya-adı alanları:", candidate_id_fields)
for f in candidate_id_fields:
    print(f"\n  {f} için ilk 5 değer:")
    for i in range(min(5, len(ds_sampled_image))):
        print(f"    {ds_sampled_image[i][f]!r}")

print("\n" + "="*70)
print("YORUM: yukarıdaki alanlardan biri gerçek COCO dosya adına/ID'sine")
print("benziyorsa (örn. '000000123456.jpg' ya da 123456 gibi bir sayı),")
print("aşağıdaki ID_FIELD değişkenini ona göre ayarla.")
print("="*70)


In [ ]:
# HÜCRE T2: COCO caption dosyasını arama
import os, json

candidate_paths = [
    f"{DRIVE_BASE}/captions_train2017.json",
    f"{DRIVE_BASE}/annotations/captions_train2017.json",
    f"{DRIVE_BASE}/coco/annotations/captions_train2017.json",
    f"{DRIVE_BASE}/ZoomInOut/captions_train2017.json",
    f"{DRIVE_BASE}/captions_val2017.json",
    f"{DRIVE_BASE}/annotations/captions_val2017.json",
    "/content/drive/MyDrive/captions_train2017.json",
    "/content/drive/MyDrive/annotations/captions_train2017.json",
]
COCO_CAPTIONS_PATH = None
for p in candidate_paths:
    if os.path.exists(p):
        COCO_CAPTIONS_PATH = p
        print(f"✅ BULUNDU: {p}")
        break

if COCO_CAPTIONS_PATH is None:
    print("❌ Standart yollarda bulunamadı.")
    print("Elinde bir COCO caption dosyası (captions_train2017.json /")
    print("captions_val2017.json, ya da Türkçe çevirisi -- ayni formatta")
    print("{\"annotations\": [{\"image_id\":.., \"caption\":..}, ...]} olmali)")
    print("varsa, aşağıya elle yolu yaz:")
    print('  COCO_CAPTIONS_PATH = "/content/drive/MyDrive/.../captions_train2017.json"')
    print("\nYoksa bu notebook otomatik olarak v4 davranışına (en-yakın-hikaye-")
    print("cümlesi) düşecek -- sadece Bölüm 2-4'teki (CV özellik zenginleştirme +")
    print("kalibrasyon) düzeltmeler uygulanacak.")


In [ ]:
# HÜCRE T3: Eşleştirme denemesi (COCO_CAPTIONS_PATH bulunduysa/ayarlandiysa)
image_id_to_captions = {}
COCO_MAPPING_READY = False
ID_FIELD = "image_id"  # <- doğrulandı: ds_sampled_image içindeki image_id, gerçek COCO image_id'siyle birebir aynı

if COCO_CAPTIONS_PATH and os.path.exists(COCO_CAPTIONS_PATH):
    with open(COCO_CAPTIONS_PATH) as f:
        coco_ann = json.load(f)
    for ann in coco_ann["annotations"]:
        image_id_to_captions.setdefault(ann["image_id"], []).append(ann["caption"])
    print(f"COCO caption dosyası yüklendi: {len(image_id_to_captions)} benzersiz görüntü, "
          f"{len(coco_ann['annotations'])} toplam caption.")

    def normalize_coco_id(raw):
        # COCO id'leri hem int hem "000000123456.jpg" formatinda olabilir
        if isinstance(raw, int):
            return raw
        s = str(raw)
        s = s.split(".")[0].lstrip("0") or "0"
        try:
            return int(s)
        except ValueError:
            return None

    if ID_FIELD is None:
        print("\n⚠️  ID_FIELD ayarlanmadı. Hücre T1'in çıktısına bakıp bu hücrenin")
        print("    başındaki ID_FIELD değişkenini ayarla, sonra bu hücreyi tekrar çalıştır.")
    else:
        n_total = len(ds_sampled_image)
        n_matched = 0
        sample_matches = []
        for i in range(n_total):
            raw_id = ds_sampled_image[i][ID_FIELD]
            norm_id = normalize_coco_id(raw_id)
            if norm_id in image_id_to_captions:
                n_matched += 1
                if len(sample_matches) < 5:
                    sample_matches.append((i, norm_id, image_id_to_captions[norm_id][0]))

        print(f"\nEşleşme oranı: {n_matched}/{n_total} (%{n_matched/n_total*100:.1f})")
        print("\nÖrnek eşleşmeler:")
        for i, cid, cap in sample_matches:
            print(f"  ds_sampled_image[{i}] (coco_id={cid}) -> {cap!r}")

        if n_matched / n_total > 0.5:
            COCO_MAPPING_READY = True
            print("\n✅ Eşleşme oranı yeterli (%50+) -- COCO caption'ları kullanılacak.")
        else:
            print("\n⚠️  Eşleşme oranı düşük -- ID_FIELD yanlış olabilir, kontrol et.")
            print("    COCO_MAPPING_READY=False kalacak, v4 davranışına düşülecek.")
else:
    print("COCO_CAPTIONS_PATH yok -- v4 davranışına (en-yakın-hikaye-cümlesi) düşülüyor.")

print(f"\nCOCO_MAPPING_READY = {COCO_MAPPING_READY}")


## 5. Zenginleştirilmiş CV Özellik Çıkarımı

**Sorun (v4'te bulundu):** `shape_feat` (16-bin kenar histogramı + aspect
ratio, 17 boyut) doygun -- farklı, örtüşmeyen, hatta object/background
karışık yamalar arasında bile cosine-benzerliği 0.997-1.000 çıkıyordu.

**Düzeltme:**
- **Doku (`texture_feat`)**: eski Sobel-gradyan histogramı (32 bin) + YENİ
  Local Binary Pattern (LBP, uniform, 10 bin) histogramı.
- **Şekil (`shape_feat`)**: kenar histogramı 16→32 bine çıkarıldı + YENİ 7
  Hu momenti (ölçek/dönüş-değişmez şekil tanımlayıcıları, log-ölçekli) +
  aspect ratio. 17 boyuttan ~40 boyuta çıkıyor.
- **Renk (`color_feat`)**: değişmedi.

In [ ]:
# HÜCRE 5b: Zenginleştirilmiş CV özellik çıkarımı (önbellekli)
def visual_loader(image_id):
    idx = int(image_id.split("_")[-1])
    return ds_sampled_image[idx]["image"].convert("RGB")


from skimage.feature import local_binary_pattern

if os.path.exists(CV_CACHE_PATH):
    print(f"⏳ Önbelleklenmiş ZENGİNLEŞTİRİLMİŞ CV özellikleri bulundu, hesaplama ATLANIYOR: {CV_CACHE_PATH}")
    with open(CV_CACHE_PATH, "rb") as f:
        cv_cache = pickle.load(f)
    n_hit = 0
    for patch in hybrid_index.image_patches:
        feats = cv_cache.get(patch.patch_id)
        if feats is not None:
            patch.color_feat, patch.shape_feat, patch.texture_feat = feats
            n_hit += 1
        else:
            patch.color_feat = patch.embedding
            patch.shape_feat = patch.embedding
            patch.texture_feat = patch.embedding
    print(f"✅ {n_hit}/{len(hybrid_index.image_patches)} yama önbellekten dolduruldu.")
else:
    print("⏳ Önbellek yok -- tüm görsel yamalar için ZENGİNLEŞTİRİLMİŞ renk/şekil/doku özellikleri hesaplanıyor (uzun sürebilir)...")
    for patch in tqdm(hybrid_index.image_patches, desc="CV öznitelikleri (zenginleştirilmiş)"):
        try:
            pil_img = visual_loader(patch.image_id)
            img_bgr = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)
            H_img, W_img = img_bgr.shape[:2]
            x1 = int(np.clip(patch.bbox[0] * (W_img / 224.0), 0, W_img - 1))
            y1 = int(np.clip(patch.bbox[1] * (H_img / 224.0), 0, H_img - 1))
            x2 = int(np.clip(patch.bbox[2] * (W_img / 224.0), 1, W_img))
            y2 = int(np.clip(patch.bbox[3] * (H_img / 224.0), 1, H_img))
            if x2 <= x1: x2 = x1 + 1
            if y2 <= y1: y2 = y1 + 1
            crop = img_bgr[y1:y2, x1:x2]

            # --- RENK: değişmedi (HSV histogramı) ---
            hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
            hist_h = cv2.calcHist([hsv], [0], None, [16], [0, 180])
            hist_s = cv2.calcHist([hsv], [1], None, [8], [0, 256])
            hist_v = cv2.calcHist([hsv], [2], None, [8], [0, 256])
            color_f = np.concatenate([hist_h, hist_s, hist_v]).flatten()
            color_f /= (np.linalg.norm(color_f) + 1e-10)

            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

            # --- DOKU: Sobel histogram (eski) + LBP histogram (YENİ) ---
            sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
            sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
            mag = np.sqrt(sobelx**2 + sobely**2)
            hist_txt, _ = np.histogram(mag, bins=32, range=(0, 255))
            hist_txt = hist_txt.astype(np.float32)

            lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
            hist_lbp, _ = np.histogram(lbp, bins=10, range=(0, 10))
            hist_lbp = hist_lbp.astype(np.float32)

            texture_f = np.concatenate([hist_txt, hist_lbp])
            texture_f /= (np.linalg.norm(texture_f) + 1e-10)

            # --- ŞEKİL: kenar histogram (16->32 bin) + Hu momentleri (YENİ) + aspect ratio ---
            edges = cv2.Canny(gray, 50, 150)
            hist_edge, _ = np.histogram(edges, bins=32, range=(0, 255))
            moments = cv2.moments(edges)
            hu = cv2.HuMoments(moments).flatten()
            hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
            hu_log = np.nan_to_num(hu_log, nan=0.0, posinf=0.0, neginf=0.0)
            aspect_ratio = float(x2 - x1) / float(y2 - y1)
            shape_f = np.concatenate([hist_edge.astype(np.float32), hu_log.astype(np.float32), [aspect_ratio]])
            shape_f /= (np.linalg.norm(shape_f) + 1e-10)

            patch.color_feat = color_f
            patch.shape_feat = shape_f
            patch.texture_feat = texture_f
        except Exception:
            patch.color_feat = patch.embedding
            patch.shape_feat = patch.embedding
            patch.texture_feat = patch.embedding

    cv_cache = {p.patch_id: (p.color_feat, p.shape_feat, p.texture_feat) for p in hybrid_index.image_patches}
    with open(CV_CACHE_PATH, "wb") as f:
        pickle.dump(cv_cache, f)
    print(f"✅ Hesaplandı ve önbelleğe kaydedildi -> {CV_CACHE_PATH} (bir sonraki oturumda ATLANACAK)")


### 5b. Mod-Bazlı Hardness Kalibrasyonu

**Sorun (v4'te bulundu):** `hardness = surface_sim - sem_sim` formülünde
tüm modlar HAM cosine-benzerlik skorlarıyla karşılaştırılıyordu. shape_feat
doygun olduğu için (surface_sim ≈ sabit ~1.0), shape modu bu yarışmayı
yapay olarak kazanıyordu (%58-62 -- gerçekten "en zor" olduğu için değil).

**Düzeltme:** her modun kendi surface_sim dağılımını (aynı görsel içindeki
rastgele yama çiftleri üzerinden) örnekleyip, hardness karşılaştırmasında
HAM skor yerine bu dağılıma göre z-normalize edilmiş skor kullanıyoruz.
Kaydedilen `hard_negative_sim` (rapor edilen değer) yine HAM/yorumlanabilir
kalıyor -- sadece mod-SEÇİMİ kalibre ediliyor.


In [ ]:
def calibrate_mode_stats(index, n_samples=3000, seed=42):
    """v5b: z-skor YERİNE tam sıralı örneklem (percentile-rank için) döndürür.
    z-skor denendiğinde std çok küçük olan modlar (shape: 0.0093) ufacık
    sapmalarda bile dev z-skorları üretip mod yarışmasını TAMAMEN ele
    geçiriyordu (renk %100 kazandı, shape/texture hiç seçilmedi). Percentile
    yaklaşımı her modun kendi dağılım şeklinden/varyansından bağımsız,
    adil bir [0,1] karşılaştırma sağlıyor."""
    rng = random.Random(seed)
    feat_attr = {"color": "color_feat", "shape": "shape_feat", "texture": "texture_feat"}
    image_ids = index.image_ids()
    stats = {}
    for mode, attr in feat_attr.items():
        sims = []
        tries = 0
        while len(sims) < n_samples and tries < n_samples * 5:
            tries += 1
            img_id = rng.choice(image_ids)
            patches = index.image_patches_of(img_id)
            if len(patches) < 2:
                continue
            a, b = rng.sample(patches, 2)
            fa, fb = getattr(a, attr, None), getattr(b, attr, None)
            if fa is None or fb is None:
                continue
            na, nb = np.linalg.norm(fa), np.linalg.norm(fb)
            if na == 0 or nb == 0:
                continue
            sims.append(float(np.dot(fa, fb) / (na * nb)))
        stats[mode] = np.sort(np.asarray(sims, dtype=np.float32))
    return stats


if os.path.exists(MODE_CALIB_PATH):
    print(f"⏳ Kaydedilmiş mod kalibrasyonu bulundu, yükleniyor: {MODE_CALIB_PATH}")
    with open(MODE_CALIB_PATH, "rb") as f:
        MODE_CALIB = pickle.load(f)
else:
    print("⏳ Mod kalibrasyonu hesaplanıyor (renk/şekil/doku için 3000'er rastgele-çift örneklemi)...")
    MODE_CALIB = calibrate_mode_stats(hybrid_index, n_samples=3000)
    with open(MODE_CALIB_PATH, "wb") as f:
        pickle.dump(MODE_CALIB, f)

print("\nMod kalibrasyonu (aynı-görsel-içi rastgele çiftlerin surface_sim dağılımı):")
for mode, sorted_sims in MODE_CALIB.items():
    print(f"  {mode:<10}: ortalama={sorted_sims.mean():.4f}  std={sorted_sims.std():.4f}  "
          f"(N={len(sorted_sims)}, percentile-rank ile kullanılacak)")


## 6. Sarmal Motor — Kalibre Edilmiş Hardness

Traversal mantığı (`txt_shift`, `img_shift`, `img_to_txt`) BİREBİR AYNI.
Tek değişiklik: mod-seçim yarışmasında (`hardness` hesabı) artık HAM
`surface_sim` yerine, Bölüm 3b'de hesaplanan mod-bazlı z-normalize edilmiş
skor kullanılıyor -- `best_surface_sim` (kaydedilen, raporlanan değer) yine
ham/yorumlanabilir kalıyor.

In [ ]:
# HÜCRE 4: MultimodalSpiralEngine

class MultimodalSpiralEngine:
    def __init__(self, index: HybridMultimodalIndex, similarity_threshold: float = 0.55,
                 max_iou_shift: float = 0.1, verbose: bool = True, min_win_size: int = 2,
                 hardness_min: float = 0.05, hard_neg_sem_thresh: float = 0.85,
                 mode_calib: Optional[dict] = None,
                 allowed_image_ids: Optional[Set[str]] = None,
                 top_k: int = 1, diversity_alpha: float = 0.0):
        self.index = index
        self.threshold = similarity_threshold
        self.max_iou_shift = max_iou_shift
        self.verbose = verbose
        self.min_win_size = min_win_size
        self.hardness_min = hardness_min
        self.hard_neg_sem_thresh = hard_neg_sem_thresh
        self._mode_field = {"semantic": "embedding", "color": "color_feat", "shape": "shape_feat", "texture": "texture_feat"}
        # v5: modlar-arasi adil karsilastirma icin z-normalizasyon istatistikleri
        # (bkz. Bolum 3b -- shape_feat'in dogmus olmasi shape modunu yapay
        # olarak kayirmasin diye).
        self.mode_calib = mode_calib or {}  # v5b: bos birak -> _calibrated ham sim'i aynen dondurur (fallback)
        # v5d: TRAIN/TEST leakage duzeltmesi -- motor, txt_to_img adiminda
        # zincirin baslangicindan BAGIMSIZ olarak TUM index'ten en yakin
        # goruseli ariyordu (train/test ayrimini bilmiyordu). allowed_image_ids
        # verilirse, arama havuzu SADECE bu kumeye kisitlanir.
        self.allowed_image_ids = set(allowed_image_ids) if allowed_image_ids is not None else None
        if self.allowed_image_ids is not None:
            self._forbidden_image_ids = set(self.index.image_ids()) - self.allowed_image_ids
        else:
            self._forbidden_image_ids = set()
        # v5i: HUB-ETKISI DUZELTMESI -- query_text/query_image eskiden saf
        # argmax'ti (deterministik, top_k=1), yani birbirine yakin embedding'li
        # farkli sorgular hep AYNI hedefe yakinsiyordu, ve 600 sekans arasinda
        # HICBIR paylasilan hafiza olmadigi icin bu yakinsama sessizce
        # tekrarlaniyordu (train havuzunun sadece %43'u kullanilmisti).
        # Simdi: top_k aday arasindan, ASIRI KULLANILANLARI cezalandirarak
        # (kullanim_sayisi * diversity_alpha kadar skor dusurerek) seciyoruz --
        # bu sayac TUM run_hybrid_loop cagrilari boyunca (yani tum sekanslar
        # arasinda) KALICI, kendi kendini duzelten adaptif bir mekanizma.
        self.top_k = top_k
        self.diversity_alpha = diversity_alpha
        self.image_use_count = {}
        self.doc_use_count = {}

    def _get_feat(self, patch, mode: str):
        return getattr(patch, self._mode_field.get(mode, "embedding"), None)

    def _calibrated(self, raw_sim: float, mode: str) -> float:
        """v5b: percentile-rank normalizasyonu (z-skor DEGIL -- z-skor, std'si
        cok kucuk olan modlari (shape) yapay sekilde ezmisti/kazandirmisti).
        mode_calib[mode], calibrate_mode_stats'tan gelen SIRALI bir dizi;
        raw_sim'in bu dizideki yuzdelik konumunu [0,1] olarak dondururuz."""
        sorted_sims = self.mode_calib.get(mode)
        if sorted_sims is None or not hasattr(sorted_sims, "__len__") or len(sorted_sims) == 0:
            return raw_sim
        rank = np.searchsorted(sorted_sims, raw_sim) / len(sorted_sims)
        return float(rank)

    def _cosine_sim(self, a, b, fallback=0.0):
        if a is None or b is None:
            return fallback
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        if na == 0 or nb == 0:
            return fallback
        return float(np.dot(a, b) / (na * nb))

    def _pick_diverse(self, candidates, use_count, key_fn):
        """v5i: top-K adaylar arasindan, kullanim sayisina gore cezalandirilmis
        skora bakarak sec (asiri kullanilan -- hub -- adaylar geri plana
        duser). Secilenin kullanim sayacini 1 artirir. Dondurulen skor HAM
        (cezasiz) skordur -- loglama/rapor tutarliligi icin."""
        if self.diversity_alpha <= 0.0 or len(candidates) <= 1:
            best = candidates[0]
        else:
            best = max(candidates, key=lambda item: item[1] - self.diversity_alpha * use_count.get(key_fn(item[0]), 0))
        key = key_fn(best[0])
        use_count[key] = use_count.get(key, 0) + 1
        return best

    def run_hybrid_loop(self, start_image_id: Optional[str] = None, loops: int = 16) -> List[HybridStepRecord]:
        log: List[HybridStepRecord] = []
        all_image_ids = self.index.image_ids()
        if self.allowed_image_ids is not None:
            all_image_ids = [i for i in all_image_ids if i in self.allowed_image_ids]
        if not all_image_ids:
            return log

        if start_image_id is None or start_image_id not in all_image_ids:
            start_image_id = secrets.choice(all_image_ids)

        visited_image_ids: Set[str] = {start_image_id}
        visited_doc_ids:   Set[str] = set()
        visited_patch_ids: Set[str] = set()

        pool = self.index.image_patches_of(start_image_id)
        obj_pool = [p for p in pool if p.type == "object"]
        cur_patch = secrets.choice(obj_pool if obj_pool else pool)
        visited_patch_ids.add(cur_patch.patch_id)

        log.append(HybridStepRecord(step=0, action="start", modality="image",
                                     image_id=start_image_id, patch_id=cur_patch.patch_id,
                                     bbox=cur_patch.bbox, similarity_mode="semantic"))
        step_n = 1

        res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                     min_win_size=self.min_win_size, threshold=self.threshold, top_k=self.top_k)
        if not res:
            res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                         min_win_size=self.min_win_size, threshold=-1.0, top_k=self.top_k)
        if not res:
            return log

        tgt_txt, sim0 = self._pick_diverse(res, self.doc_use_count, lambda p: p.doc_id)
        visited_doc_ids.add(tgt_txt.doc_id)
        visited_patch_ids.add(tgt_txt.patch_id)
        hn_txt = self.index.find_hard_negative_text(tgt_txt, visited_doc_ids,
                                                      hardness_min=self.hardness_min,
                                                      hard_neg_sem_thresh=self.hard_neg_sem_thresh)
        log.append(HybridStepRecord(step=step_n, action="img_to_txt", modality="text",
                                     doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                     text_content=tgt_txt.text_content, similarity=sim0, similarity_mode="semantic",
                                     hard_negative_txt_patch_id=hn_txt[0] if hn_txt else None,
                                     hard_negative_txt_sim=hn_txt[1] if hn_txt else None,
                                     hard_negative_txt_mode=hn_txt[2] if hn_txt else None))
        step_n += 1

        for lp in range(1, loops + 1):
            txt_pool = [p for p in self.index.text_patches_of(tgt_txt.doc_id)
                        if p.patch_id not in visited_patch_ids and p.win_size >= self.min_win_size
                        and self.index._is_quality_text(p.text_content)]
            if not txt_pool:
                txt_pool = [p for p in self.index.text_patches_of(tgt_txt.doc_id) if p.patch_id not in visited_patch_ids]
            if txt_pool:
                tgt_txt = max(txt_pool, key=lambda p: self._cosine_sim(tgt_txt.embedding, p.embedding))
                visited_patch_ids.add(tgt_txt.patch_id)
                log.append(HybridStepRecord(step=step_n, action="txt_shift", modality="text",
                                             doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                             text_content=tgt_txt.text_content, similarity_mode="semantic"))
                step_n += 1

            # v5d: ziyaret edilenlere EK olarak, izin verilen havuzun DISINDAKI
            # tum gorselleri de haric tut -- boylece train sekansi test
            # gorseline (ya da tam tersi) SICRAYAMAZ.
            img_res = self.index.query_image(tgt_txt.embedding, exclude_image_ids=visited_image_ids | self._forbidden_image_ids, top_k=self.top_k)
            if not img_res:
                break
            best_img, best_score = self._pick_diverse(img_res, self.image_use_count, lambda p: p.image_id)
            visited_image_ids.add(best_img.image_id)
            visited_patch_ids.add(best_img.patch_id)
            log.append(HybridStepRecord(step=step_n, action="txt_to_img", modality="image",
                                         image_id=best_img.image_id, patch_id=best_img.patch_id,
                                         bbox=best_img.bbox, similarity=best_score, similarity_mode="semantic"))
            step_n += 1
            cur_patch = best_img

            MIN_OBJ = 0.4
            img_pool_raw = self.index.image_patches_of(cur_patch.image_id)
            valid_pool = [p for p in img_pool_raw if p.patch_id not in visited_patch_ids
                          and cur_patch.iou(p) <= self.max_iou_shift and p.objectness_score >= MIN_OBJ]
            if not valid_pool:
                valid_pool = sorted([p for p in img_pool_raw if p.patch_id not in visited_patch_ids],
                                     key=lambda x: x.objectness_score, reverse=True)[:5]

            if valid_pool:
                best_mode, best_patch = None, None
                best_surface_sim, best_sem_sim, best_hardness = 0.0, 0.0, -999.0
                for m in ("color", "shape", "texture"):
                    q_feat = self._get_feat(cur_patch, m)
                    if q_feat is None:
                        continue
                    candidate = max(valid_pool, key=lambda p: self._cosine_sim(q_feat, self._get_feat(p, m), fallback=-1.0))
                    surface_sim = self._cosine_sim(q_feat, self._get_feat(candidate, m), fallback=-1.0)
                    sem_sim = self._cosine_sim(cur_patch.embedding, candidate.embedding)
                    # v5: hardness karsilastirmasi z-normalize edilmis surface_sim ile yapiliyor
                    # (raporlanan best_surface_sim yine HAM deger olarak kaydediliyor).
                    surface_sim_z = self._calibrated(surface_sim, m)
                    hardness = surface_sim_z - sem_sim
                    if hardness > best_hardness:
                        best_mode, best_patch = m, candidate
                        best_surface_sim, best_sem_sim, best_hardness = surface_sim, sem_sim, hardness

                # NOT: best_hardness artik z-skor farkindan geldigi icin, hardness_min
                # esigi de v5'te yeniden kalibre edilmis olmali (asagidaki HUCRE E1'e bak).
                if best_patch is not None and best_hardness >= self.hardness_min:
                    mode = best_mode
                    tgt_img = best_patch
                    score = best_surface_sim
                    computed_label = "hard_negative" if best_sem_sim < self.hard_neg_sem_thresh else "positive"
                else:
                    mode = "semantic"
                    tgt_img = secrets.choice(valid_pool)
                    score = self._cosine_sim(cur_patch.embedding, tgt_img.embedding)
                    computed_label = "positive"
            else:
                mode = "semantic"
                score = 0.0
                computed_label = "positive"
                others = [p for p in img_pool_raw if p.patch_id != cur_patch.patch_id]
                tgt_img = min(others, key=lambda p: cur_patch.iou(p)) if others else cur_patch

            visited_patch_ids.add(tgt_img.patch_id)
            log.append(HybridStepRecord(step=step_n, action="img_shift", modality="image",
                                         image_id=tgt_img.image_id, patch_id=tgt_img.patch_id,
                                         bbox=tgt_img.bbox, similarity=score,
                                         similarity_mode=mode, verified_label=computed_label))
            step_n += 1
            cur_patch = tgt_img

            res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                         min_win_size=self.min_win_size, threshold=self.threshold, top_k=self.top_k)
            if not res:
                res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                             min_win_size=self.min_win_size, threshold=-1.0, top_k=self.top_k)
            if not res:
                break
            tgt_txt, sim = self._pick_diverse(res, self.doc_use_count, lambda p: p.doc_id)
            visited_doc_ids.add(tgt_txt.doc_id)
            visited_patch_ids.add(tgt_txt.patch_id)
            hn_txt = self.index.find_hard_negative_text(tgt_txt, visited_doc_ids,
                                                          hardness_min=self.hardness_min,
                                                          hard_neg_sem_thresh=self.hard_neg_sem_thresh)
            log.append(HybridStepRecord(step=step_n, action="img_to_txt", modality="text",
                                         doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                         text_content=tgt_txt.text_content, similarity=sim, similarity_mode="semantic",
                                         hard_negative_txt_patch_id=hn_txt[0] if hn_txt else None,
                                         hard_negative_txt_sim=hn_txt[1] if hn_txt else None,
                                         hard_negative_txt_mode=hn_txt[2] if hn_txt else None))
            step_n += 1

        return log


### 6b. hardness_min Yeniden Kalibrasyonu (z-skor ölçeği için)

`hardness` artık `z_normalize(surface_sim) - sem_sim` -- eski ham-cosine
ölçeğindeki `hardness_min=0.05` bu yeni ölçekte anlamsız olabilir (z-skorlar
tipik olarak [-3, 3] aralığında, eskisi [-1, 1] aralığındaydı). v3/v4'te
yapıldığı gibi (bkz. Karar #10: 0.15/0.75 çok sıkıydı, 0.05/0.85 iyi verim
verdi), burada da küçük bir deneme setiyle birkaç eşiği hızlıca tarayıp
verimi (kaç sekansta doğrulanmış hard-negative bulunuyor) ölçüyoruz.

In [ ]:
# HÜCRE E1: hardness_min kalibrasyonu (30 deneme sekansıyla hızlı tarama)
_calib_engine_test = MultimodalSpiralEngine(index=hybrid_index, similarity_threshold=SIM_THRESHOLD if "SIM_THRESHOLD" in globals() else 0.55,
                                             max_iou_shift=0.1, min_win_size=2, verbose=False,
                                             mode_calib=MODE_CALIB)
_test_image_ids = hybrid_index.image_ids()[:200]

for _hmin in [0.0, 0.1, 0.3, 0.5, 1.0]:
    _calib_engine_test.hardness_min = _hmin
    _n_hn = 0
    _n_total_shift = 0
    _mode_counts = {}
    for _ in range(30):
        chain = _calib_engine_test.run_hybrid_loop(start_image_id=random.choice(_test_image_ids), loops=16)
        for step in chain:
            if step.action == "img_shift":
                _n_total_shift += 1
                if step.verified_label == "hard_negative":
                    _n_hn += 1
                    _mode_counts[step.similarity_mode] = _mode_counts.get(step.similarity_mode, 0) + 1
    yield_pct = _n_hn / max(_n_total_shift, 1) * 100
    print(f"hardness_min={_hmin:<5} -> {_n_hn}/{_n_total_shift} hard-negative (%{yield_pct:.1f}), mod dağılımı: {_mode_counts}")

print("\nEn dengeli mod dağılımı + makul verim (%15-30 arası, v4\'teki %24.3\'e yakın)")
print("veren hardness_min değerini seç, aşağıdaki HARDNESS_MIN değişkenine yaz.")


In [ ]:
# HÜCRE E2: seçilen esigi sabitle (yukarıdaki taramaya bakip elle ayarla)
HARDNESS_MIN = 0.1  # <-- Hücre E1 çıktısına göre ayarla
HARD_NEG_SEM_THRESH = 0.85  # v4'teki gibi kalsın, sem_sim ölçeği değişmedi
print(f"HARDNESS_MIN={HARDNESS_MIN}, HARD_NEG_SEM_THRESH={HARD_NEG_SEM_THRESH} olarak ayarlandı.")


## 7. COCO Caption Entegrasyonu (SADECE COCO_MAPPING_READY=True ise gerekli)

Bölüm 1'de eşleşme kurulduysa: (a) her `coco_image_N` için orijinal COCO
id'sini önceden hesapla, (b) caption'ları Qwen3-VL-Embedding-2B ile embed
et (bu adım GPU gerektirir ve modelin yeniden yüklenmesi anlamına gelir --
v3/v4 pipeline'ında bu model hiç yüklenmiyordu, sadece önceden hesaplanmış
embedding'lerle çalışılıyordu).

COCO_MAPPING_READY=False ise bu bölüm otomatik atlanır, hiçbir şey bozulmaz.

In [ ]:
# HÜCRE C0: coco_image_N -> orijinal COCO id eşleme tablosu
patch_image_id_to_coco_id = {}
if COCO_MAPPING_READY:
    for i in range(len(ds_sampled_image)):
        raw_id = ds_sampled_image[i][ID_FIELD]
        norm_id = normalize_coco_id(raw_id)
        patch_image_id_to_coco_id[f"coco_image_{i}"] = norm_id
    print(f"✅ {len(patch_image_id_to_coco_id)} görüntü için coco_id eşlemesi hazır.")
else:
    print("COCO_MAPPING_READY=False -- eşleme tablosu boş bırakıldı.")


In [ ]:
# HÜCRE C1: COCO caption\'larını Qwen3-VL-Embedding-2B ile embed et
coco_caption_embeddings = {}  # {coco_id: [(caption_text, embedding), ...]}

if COCO_MAPPING_READY:
    if os.path.exists(COCO_CAPTIONS_EMB_PATH):
        print(f"⏳ Önceden embed edilmiş COCO caption\'ları bulundu, yükleniyor: {COCO_CAPTIONS_EMB_PATH}")
        with open(COCO_CAPTIONS_EMB_PATH, "rb") as f:
            coco_caption_embeddings = pickle.load(f)
    else:
        print("⏳ Qwen3-VL-Embedding-2B yükleniyor (GPU gerektirir, biraz sürer)...")
        from transformers import AutoModel, AutoProcessor

        _qwen_model_id = "Qwen/Qwen3-VL-Embedding-2B"
        _qwen_model = AutoModel.from_pretrained(_qwen_model_id, trust_remote_code=True,
                                                 torch_dtype=torch.float16).to(device)
        _qwen_processor = AutoProcessor.from_pretrained(_qwen_model_id, trust_remote_code=True)
        print("✅ Model yüklendi.")

        def _last_token_pool(last_hidden_states, attention_mask):
            left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
            if left_padding:
                return last_hidden_states[:, -1]
            seq_lens = attention_mask.sum(dim=1) - 1
            batch_size = last_hidden_states.shape[0]
            return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), seq_lens]

        def _embed_texts(texts, batch_size=32):
            all_embs = []
            for start in range(0, len(texts), batch_size):
                batch = texts[start:start + batch_size]
                inputs = _qwen_processor(text=batch, padding=True, truncation=True,
                                          max_length=64, return_tensors="pt").to(device)
                with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    out = _qwen_model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask,
                                       output_hidden_states=True, return_dict=True)
                    last_hidden = out.hidden_states[-1] if hasattr(out, "hidden_states") else out[0]
                    emb = _last_token_pool(last_hidden.float(), inputs.attention_mask)
                    emb = F.normalize(emb, p=2, dim=-1)
                all_embs.append(emb.cpu().numpy())
            return np.concatenate(all_embs, axis=0)

        # SADECE bizim 2000 örneklem görselimizin caption'larını embed et
        # (image_id_to_captions TÜM COCO'yu -- 118K görsel -- içeriyor, gereksiz
        # hesaplamadan kaçınmak için patch_image_id_to_coco_id ile filtrele).
        _needed_coco_ids = set(patch_image_id_to_coco_id.values())
        all_pairs_flat = [(coco_id, cap) for coco_id, caps in image_id_to_captions.items()
                           if coco_id in _needed_coco_ids for cap in caps]
        all_texts_flat = [p[1] for p in all_pairs_flat]
        print(f"({len(image_id_to_captions)} COCO görselinden sadece {len(_needed_coco_ids)} tanesi "
              f"bizim örneklemimizde -- {len(all_texts_flat)} caption embed edilecek, "
              f"591753 değil.)")
        print(f"⏳ {len(all_texts_flat)} caption embed ediliyor...")
        all_embs_flat = _embed_texts(all_texts_flat, batch_size=64)

        for (coco_id, cap), emb in zip(all_pairs_flat, all_embs_flat):
            coco_caption_embeddings.setdefault(coco_id, []).append((cap, emb))

        with open(COCO_CAPTIONS_EMB_PATH, "wb") as f:
            pickle.dump(coco_caption_embeddings, f)
        print(f"✅ Kaydedildi -> {COCO_CAPTIONS_EMB_PATH}")

        del _qwen_model, _qwen_processor
        torch.cuda.empty_cache()

    print(f"\n{len(coco_caption_embeddings)} görüntü için caption embedding\'i hazır.")
else:
    print("COCO_MAPPING_READY=False -- bu hücre atlanıyor, v4 davranışına (en-yakın-hikaye-cümlesi) devam edilecek.")


## 8. Ayarlar + Üretim Fonksiyonları

`build_pairs_from_sequences` DEĞİŞMEDİ (zincirden çift çıkarma mantığı
aynı). Sadece `attach_embeddings` değişti: COCO_MAPPING_READY=True ise,
pozitif metin embedding'ini "en yakın hikaye cümlesi" yerine o görselin
GERÇEK COCO caption'ıyla (varsa) değiştiriyor.

In [ ]:
# HÜCRE 6: Ayarlar + train/test split + tüm üretim fonksiyonları

NUM_SEQUENCES        = 600  # v5e: 300->600, cesitlilik + hacim icin
LOOPS_PER_SEQUENCE   = 16
MIN_CHAIN_LENGTH     = 16
SIM_THRESHOLD        = 0.55
TRAIN_RATIO          = 0.75  # v5e: test havuzunu 400->500 gorsele cikardik (cesitlilik icin)

TEST_NUM_SEQUENCES   = 600   # v5e: 200->600, daha fazla benzersiz gorsele ulasma sansi icin
TEST_TARGET_SAMPLES  = 400   # v5e: 500->400, 500 gorselik havuzda gercekci/ulasilabilir hedef

VAL_RATIO            = 0.15  # v5h: train havuzunun %15'i erken-durdurma/model-secimi icin ayrilir
VAL_NUM_SEQUENCES    = 300
VAL_TARGET_SAMPLES   = 150

TOP_K              = 8     # GERI DONULDU: v5i degeri (v5j denemesi 14 idi)
DIVERSITY_ALPHA    = 0.03  # GERI DONULDU: v5i degeri (v5j denemesi 0.06 idi)


def make_image_split(hybrid_index, train_ratio=0.8, seed=42):
    rng = random.Random(seed)
    all_ids = hybrid_index.image_ids()
    rng.shuffle(all_ids)
    split_point = int(len(all_ids) * train_ratio)
    return set(all_ids[:split_point]), set(all_ids[split_point:])


def carve_val_split(train_image_ids, val_ratio=0.15, seed=43):
    """v5h: train havuzunun bir kismini erken-durdurma / model-secimi icin
    ayirir. Test'e HIC dokunmaz -- sadece train'i (final_train, val) olarak
    ikiye boler. sorted() ile deterministik (set sirasina guvenmiyoruz)."""
    rng = random.Random(seed)
    all_ids = sorted(train_image_ids)
    rng.shuffle(all_ids)
    split_point = int(len(all_ids) * (1 - val_ratio))
    return set(all_ids[:split_point]), set(all_ids[split_point:])


def generate_sequences(hybrid_index, engine, image_ids, num_sequences, loops, min_chain_length):
    """v5j: GARANTI-KAPSAMLI baslangic secimi -- eskiden her cekilis
    BAGIMSIZ rastgeleydi (random.choice), yani 600 cekiliste havuzun
    (1275) sadece ~%38'i (kupon toplama beklentisi) baslangic olarak
    kullanilirdi. Simdi havuz once KARISTIRILIP SIRAYLA gezilir -- ilk
    min(num_sequences, len(image_ids)) cekiliste HER goruntu TAM 1 KEZ
    baslangic olur (garantili), fazlasi icin bastan tekrar baslanir."""
    image_ids = list(image_ids)
    _rng_starts = random.Random(1234)
    shuffled = image_ids[:]
    _rng_starts.shuffle(shuffled)
    sequences = []
    for i in tqdm(range(num_sequences), desc="Sekans üretiliyor"):
        start_img = shuffled[i % len(shuffled)]
        chain = engine.run_hybrid_loop(start_image_id=start_img, loops=loops)
        if len(chain) >= min_chain_length:
            sequences.append(chain)
    print(f"✅ {len(sequences)}/{num_sequences} sekans üretildi (min_chain_length={min_chain_length} filtresinden geçti)")
    return sequences


def build_pairs_from_sequences(sequences, sim_threshold):
    """cross-modal pozitif çiftler + SADECE verified_label=='hard_negative' olan
    img_shift'lerden alınan hard negative'ler."""
    records = []
    n_with_hn = 0
    for chain in sequences:
        for i, step in enumerate(chain):
            if step.action not in ("img_to_txt", "txt_to_img") or i == 0:
                continue
            prev = chain[i - 1]
            img_step, txt_step = (prev, step) if step.action == "img_to_txt" else (step, prev)
            if img_step.modality != "image" or txt_step.modality != "text":
                continue
            cross_sim = step.similarity if step.similarity else 0.0
            if cross_sim < sim_threshold:
                continue

            record = {
                "txt_patch_id": txt_step.patch_id, "img_patch_id": img_step.patch_id,
                "cross_modal_sim": cross_sim,
                "hard_negative_patch_id": None, "hard_negative_sim": None, "hard_negative_mode": None,
                # Fikir 2: metin-tarafı hard-negative (sadece txt_step bir img_to_txt
                # adımıysa dolu olur -- txt_to_img çiftlerinde txt_step bir txt_shift
                # olduğu için bu alanlar None kalır, kapsam ~%50 civarında olur).
                "hard_negative_txt_patch_id": getattr(txt_step, "hard_negative_txt_patch_id", None),
                "hard_negative_txt_sim": getattr(txt_step, "hard_negative_txt_sim", None),
                "hard_negative_txt_mode": getattr(txt_step, "hard_negative_txt_mode", None),
            }
            if i + 1 < len(chain) and chain[i + 1].action == "img_shift":
                shift_step = chain[i + 1]
                if (shift_step.image_id == img_step.image_id
                        and getattr(shift_step, "verified_label", "positive") == "hard_negative"):
                    record["hard_negative_patch_id"] = shift_step.patch_id
                    record["hard_negative_sim"] = shift_step.similarity
                    record["hard_negative_mode"] = shift_step.similarity_mode
                    n_with_hn += 1
            records.append(record)

    print(f"Toplam cross-modal çift: {len(records)} | Doğrulanmış hard-negative içeren: {n_with_hn} (%{n_with_hn/max(len(records),1)*100:.1f})")
    return records


def attach_embeddings(records, hybrid_index):
    txt_lookup = {p.patch_id: p.embedding for p in hybrid_index.text_patches}
    img_lookup = {p.patch_id: p.embedding for p in hybrid_index.image_patches}
    img_id_lookup = {p.patch_id: p.image_id for p in hybrid_index.image_patches}  # v5: coco caption eşlemesi için
    final = []
    n_with_hn_txt = 0
    n_coco_caption_used = 0
    for r in records:
        txt_emb = txt_lookup.get(r["txt_patch_id"])
        img_emb = img_lookup.get(r["img_patch_id"])
        if txt_emb is None or img_emb is None:
            continue

        # v5: COCO caption bulunabiliyorsa, pozitif metni "en yakın hikaye
        # cümlesi" yerine görselin GERÇEK caption'ıyla değiştir.
        if COCO_MAPPING_READY:
            img_image_id = img_id_lookup.get(r["img_patch_id"])
            coco_id = patch_image_id_to_coco_id.get(img_image_id)
            if coco_id is not None and coco_id in coco_caption_embeddings:
                _cap_text, _cap_emb = random.choice(coco_caption_embeddings[coco_id])
                txt_emb = _cap_emb
                n_coco_caption_used += 1

        hn_emb = img_lookup.get(r["hard_negative_patch_id"]) if r["hard_negative_patch_id"] is not None else None
        hn_txt_emb = txt_lookup.get(r["hard_negative_txt_patch_id"]) if r.get("hard_negative_txt_patch_id") is not None else None
        if hn_txt_emb is not None:
            n_with_hn_txt += 1
        final.append({
            "txt_embedding": np.asarray(txt_emb, dtype=np.float32),
            "img_embedding": np.asarray(img_emb, dtype=np.float32),
            "cross_modal_sim": r["cross_modal_sim"],
            "hard_negative_embedding": np.asarray(hn_emb, dtype=np.float32) if hn_emb is not None else None,
            "hard_negative_sim": r["hard_negative_sim"],
            "hard_negative_mode": r["hard_negative_mode"],
            "hard_negative_txt_embedding": np.asarray(hn_txt_emb, dtype=np.float32) if hn_txt_emb is not None else None,
            "hard_negative_txt_sim": r.get("hard_negative_txt_sim"),
        })
    print(f"Embedding'i tam bulunan final kayıt: {len(final)} (metin-tarafı hard-negative içeren: {n_with_hn_txt}, %{n_with_hn_txt/max(len(final),1)*100:.1f})")
    if COCO_MAPPING_READY:
        print(f"COCO caption ile değiştirilen pozitif metin sayısı: {n_coco_caption_used} (%{n_coco_caption_used/max(len(final),1)*100:.1f})")
    return final


def generate_test_pairs(hybrid_index, engine, test_image_ids, num_sequences, loops,
                         min_chain_length, sim_threshold, target_samples):
    """v5f: TEST HAVUZU CESITLILIGI + DAGILIM TUTARLILIGI DUZELTMESI.
    Eskiden iki sorun vardi: (1) ayni gorsel test kumesinde defalarca
    tekrarlanabiliyordu (R@1'i yapay tavanliyordu -- v5e'de duzeltildi,
    her gorselden EN FAZLA 1 ornek); (2) SADECE dogrulanmis hard-negative
    iceren kayitlar tutuluyordu -- bu, test kumesini train'den (hard-negative
    orani ~%17) FARKLI bir dagilima (test'te %100) zorluyordu VE havuzu
    (17896 -> 2928 -> sadece 117 benzersiz gorsel) cok daralttiginin asil
    nedeniydi. Simdi test, train ile AYNI (filtresiz) dagilimdan orneklendigi
    icin havuz cok daha genis, benzersiz gorsele ulasmak cok daha kolay."""
    test_sequences = []
    test_image_ids = list(test_image_ids)
    _rng_starts = random.Random(1234)
    shuffled = test_image_ids[:]
    _rng_starts.shuffle(shuffled)
    for i in tqdm(range(num_sequences), desc="Test sekansları"):
        start_img = shuffled[i % len(shuffled)]
        chain = engine.run_hybrid_loop(start_image_id=start_img, loops=loops)
        if len(chain) >= min_chain_length:
            test_sequences.append(chain)

    raw_records = build_pairs_from_sequences(test_sequences, sim_threshold)
    random.shuffle(raw_records)

    img_id_lookup = {p.patch_id: p.image_id for p in hybrid_index.image_patches}
    seen_images = set()
    deduped_records = []
    for r in raw_records:
        src_img_id = img_id_lookup.get(r["img_patch_id"])
        if src_img_id in seen_images:
            continue
        seen_images.add(src_img_id)
        deduped_records.append(r)
        if len(deduped_records) >= target_samples:
            break

    if len(deduped_records) < target_samples:
        print(f"⚠️  UYARI: hedef {target_samples} idi ama sadece {len(deduped_records)} BENZERSİZ "
              f"görselden geçerli hard-negative çift bulunabildi. Test havuzunu (test_image_ids "
              f"oranı / num_sequences) büyütmen gerekebilir.")
    print(f"Benzersiz kaynak görsel sayısı (tekrarsız): {len(deduped_records)} / hedef {target_samples}")

    final_pairs = attach_embeddings(deduped_records, hybrid_index)
    mode_counts = {}
    for p in final_pairs:
        m = p["hard_negative_mode"]
        mode_counts[m] = mode_counts.get(m, 0) + 1
    print("Mod dağılımı:", mode_counts)
    return final_pairs


train_image_ids, test_image_ids = make_image_split(hybrid_index, TRAIN_RATIO)
train_image_ids, val_image_ids = carve_val_split(train_image_ids, VAL_RATIO, seed=43)
print(f"Train görsel sayısı: {len(train_image_ids)} | Val görsel sayısı: {len(val_image_ids)} | Test görsel sayısı: {len(test_image_ids)}")


## 9. Train + Test Üretimi (v5 -- kalibre edilmiş motor + yeni CV özellikleri)

In [ ]:
# HÜCRE 7: Train + val + test üretimi (üçü de AYRI AYRI önbellek kontrolü)
# v5i: motor artik hub-etkisi duzeltmesiyle (top_k + diversity_alpha)
# calisiyor -- eski (v5h) onbellekler bu yuzden ARTIK GECERSIZ, hepsi
# sifirdan uretilecek (yeni v5i isimleriyle, eskiler hic aranmiyor/silinmiyor).

if os.path.exists(TRAIN_PAIRS_PATH) and os.path.exists(CHAIN_RECORDS_PATH):
    print("⏳ Train zaten var, diskten yükleniyor...")
    with open(TRAIN_PAIRS_PATH, "rb") as f:
        train_pairs = pickle.load(f)["train_pairs"]
    print(f"✅ Train yüklendi: {len(train_pairs)}")
    print("⚠️  NOT: train_sequences bu yolla YÜKLENEMEZ (sadece embedding'ler kaydedildi).")
    print("    SACL bölümü, train_chain_records önbelleğinden (CHAIN_RECORDS_PATH) yüklenecek.")
elif os.path.exists(TRAIN_PAIRS_PATH) and not os.path.exists(CHAIN_RECORDS_PATH):
    print("⚠️  train_pairs var ama train_chain_records yok (önceki çalıştırma yarıda kalmış).")
    print("    Tutarlılık için train_pairs önbelleği YOK SAYILIYOR, ikisi birlikte sıfırdan üretiliyor...")
    train_engine = MultimodalSpiralEngine(index=hybrid_index, similarity_threshold=SIM_THRESHOLD,
                                           max_iou_shift=0.1, min_win_size=2, verbose=False,
                                           hardness_min=HARDNESS_MIN, hard_neg_sem_thresh=HARD_NEG_SEM_THRESH,
                                           mode_calib=MODE_CALIB, allowed_image_ids=train_image_ids,
                                           top_k=TOP_K, diversity_alpha=DIVERSITY_ALPHA)
    train_sequences = generate_sequences(hybrid_index, train_engine, train_image_ids,
                                          NUM_SEQUENCES, LOOPS_PER_SEQUENCE, MIN_CHAIN_LENGTH)
    train_records = build_pairs_from_sequences(train_sequences, SIM_THRESHOLD)
    train_pairs = attach_embeddings(train_records, hybrid_index)
    with open(TRAIN_PAIRS_PATH, "wb") as f:
        pickle.dump({"train_pairs": train_pairs, "config": {
            "num_sequences": NUM_SEQUENCES, "loops_per_sequence": LOOPS_PER_SEQUENCE,
            "sim_threshold": SIM_THRESHOLD, "train_ratio": TRAIN_RATIO,
            "top_k": TOP_K, "diversity_alpha": DIVERSITY_ALPHA}}, f)
    print(f"✅ Kaydedildi: train={len(train_pairs)} -> {TRAIN_PAIRS_PATH}")

    _train_touched = len(set(train_engine.image_use_count.keys()))
    _train_pool = len(train_image_ids)
    print(f"[HUB KONTROLU] train motoru {_train_touched}/{_train_pool} farkli gorsele dokundu (%{_train_touched/_train_pool*100:.1f})")
else:
    print("⏳ Train yok, sıfırdan üretiliyor...")
    train_engine = MultimodalSpiralEngine(index=hybrid_index, similarity_threshold=SIM_THRESHOLD,
                                           max_iou_shift=0.1, min_win_size=2, verbose=False,
                                           hardness_min=HARDNESS_MIN, hard_neg_sem_thresh=HARD_NEG_SEM_THRESH,
                                           mode_calib=MODE_CALIB, allowed_image_ids=train_image_ids,
                                           top_k=TOP_K, diversity_alpha=DIVERSITY_ALPHA)
    train_sequences = generate_sequences(hybrid_index, train_engine, train_image_ids,
                                          NUM_SEQUENCES, LOOPS_PER_SEQUENCE, MIN_CHAIN_LENGTH)
    train_records = build_pairs_from_sequences(train_sequences, SIM_THRESHOLD)
    train_pairs = attach_embeddings(train_records, hybrid_index)
    with open(TRAIN_PAIRS_PATH, "wb") as f:
        pickle.dump({"train_pairs": train_pairs, "config": {
            "num_sequences": NUM_SEQUENCES, "loops_per_sequence": LOOPS_PER_SEQUENCE,
            "sim_threshold": SIM_THRESHOLD, "train_ratio": TRAIN_RATIO,
            "top_k": TOP_K, "diversity_alpha": DIVERSITY_ALPHA}}, f)
    print(f"✅ Kaydedildi: train={len(train_pairs)} -> {TRAIN_PAIRS_PATH}")

    # v5i: hub-kontrolunu HEMEN dogrula -- duzeltme gercekten ise yaradi mi?
    _train_touched = len(set(train_engine.image_use_count.keys()))
    _train_pool = len(train_image_ids)
    print(f"[HUB KONTROLU] train motoru {_train_touched}/{_train_pool} farkli gorsele dokundu (%{_train_touched/_train_pool*100:.1f})")

if os.path.exists(VAL_PAIRS_PATH):
    print("⏳ Val zaten var, diskten yükleniyor...")
    with open(VAL_PAIRS_PATH, "rb") as f:
        val_pairs = pickle.load(f)["val_pairs"]
    print(f"✅ Val yüklendi: {len(val_pairs)}")
else:
    print("⏳ Val yok, sıfırdan üretiliyor...")
    val_engine = MultimodalSpiralEngine(index=hybrid_index, similarity_threshold=SIM_THRESHOLD,
                                         max_iou_shift=0.1, min_win_size=2, verbose=False,
                                         hardness_min=HARDNESS_MIN, hard_neg_sem_thresh=HARD_NEG_SEM_THRESH,
                                         mode_calib=MODE_CALIB, allowed_image_ids=val_image_ids,
                                         top_k=TOP_K, diversity_alpha=DIVERSITY_ALPHA)
    val_pairs = generate_test_pairs(hybrid_index, val_engine, val_image_ids,
                                     VAL_NUM_SEQUENCES, LOOPS_PER_SEQUENCE, MIN_CHAIN_LENGTH,
                                     SIM_THRESHOLD, VAL_TARGET_SAMPLES)
    with open(VAL_PAIRS_PATH, "wb") as f:
        pickle.dump({"val_pairs": val_pairs, "config": {}}, f)
    print(f"✅ Kaydedildi: val={len(val_pairs)} -> {VAL_PAIRS_PATH}")

    _val_touched = len(set(val_engine.image_use_count.keys()))
    _val_pool = len(val_image_ids)
    print(f"[HUB KONTROLU] val motoru {_val_touched}/{_val_pool} farkli gorsele dokundu (%{_val_touched/_val_pool*100:.1f})")

if os.path.exists(TEST_PAIRS_PATH):
    print("⏳ Test zaten var, diskten yükleniyor...")
    with open(TEST_PAIRS_PATH, "rb") as f:
        test_pairs = pickle.load(f)["test_pairs"]
    print(f"✅ Test yüklendi: {len(test_pairs)}")
else:
    print("⏳ Test yok, sıfırdan üretiliyor (filtre YOK + dedup)...")
    test_engine = MultimodalSpiralEngine(index=hybrid_index, similarity_threshold=SIM_THRESHOLD,
                                          max_iou_shift=0.1, min_win_size=2, verbose=False,
                                          hardness_min=HARDNESS_MIN, hard_neg_sem_thresh=HARD_NEG_SEM_THRESH,
                                          mode_calib=MODE_CALIB, allowed_image_ids=test_image_ids,
                                          top_k=TOP_K, diversity_alpha=DIVERSITY_ALPHA)
    test_pairs = generate_test_pairs(hybrid_index, test_engine, test_image_ids,
                                      TEST_NUM_SEQUENCES, LOOPS_PER_SEQUENCE, MIN_CHAIN_LENGTH,
                                      SIM_THRESHOLD, TEST_TARGET_SAMPLES)
    with open(TEST_PAIRS_PATH, "wb") as f:
        pickle.dump({"test_pairs": test_pairs, "config": {
            "num_sequences": TEST_NUM_SEQUENCES, "loops_per_sequence": LOOPS_PER_SEQUENCE,
            "sim_threshold": SIM_THRESHOLD, "top_k": TOP_K, "diversity_alpha": DIVERSITY_ALPHA}}, f)
    print(f"✅ Kaydedildi: test={len(test_pairs)} -> {TEST_PAIRS_PATH}")

    _test_touched = len(set(test_engine.image_use_count.keys()))
    _test_pool = len(test_image_ids)
    print(f"[HUB KONTROLU] test motoru {_test_touched}/{_test_pool} farkli gorsele dokundu (%{_test_touched/_test_pool*100:.1f})")


## 9b. Gerçek Semantik Hard-Negative (COCO Kategori) -- Güvenlik Kontrollü

Şu ana kadar hard-negative'i CV yaklaşıklamasıyla (aynı fotoğraf içinde
renk/şekil/doku benzerliği) buluyorduk. Burada, COCO'nun kendi nesne
kategori etiketlerini (`ds_sampled_image[i]["objects"]["category"]`)
kullanarak FARKLI bir fotoğraftan ama AYNI kategoriden gerçek bir
semantik hard-negative eklemeyi deniyoruz -- daha güvenilir bir sinyal
olması beklenir.

**Önemli:** Patch bbox'ları (önceden hesaplanmış embedding dosyasından)
ile COCO'nun ham nesne bbox'larının AYNI koordinat sisteminde olduğu
doğrulanmadı -- bu yüzden aşağıda İKİ AYRI çalışma-zamanı kontrolü var.
Herhangi biri başarısız olursa özellik OTOMATİK VE GÜVENLİ şekilde devre
dışı kalır, mevcut CV-tabanlı yöntem hiç değişmeden kullanılmaya devam
eder -- motor koduna hiç dokunulmadı, sadece train_pairs'e (varsa)
opsiyonel bir alan ekleniyor.

In [ ]:
# HÜCRE R1: gercek semantik hard-negative (COCO kategori) -- guvenlik kontrollu
REALNEG_ENABLED = False

image_id_to_objects = {}
for _i in range(len(ds_sampled_image)):
    _key = f"coco_image_{_i}"
    _objs = ds_sampled_image[_i]["objects"]
    image_id_to_objects[_key] = list(zip(_objs["bbox"], _objs["category"]))

def _iou_xyxy(b1, b2):
    x1 = max(b1[0], b2[0]); y1 = max(b1[1], b2[1])
    x2 = min(b1[2], b2[2]); y2 = min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter == 0:
        return 0.0
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    return inter / (a1 + a2 - inter + 1e-10)

# KONTROL 1: patch bbox'lari COCO nesne bbox'lariyla makul oranda ortusuyor mu?
patch_category = {}
_checked = 0
_matched = 0
for p in hybrid_index.image_patches:
    if p.type != "object":
        continue
    objs = image_id_to_objects.get(p.image_id, [])
    if not objs:
        continue
    best_iou = 0.0
    best_cat = None
    for bbox, cat in objs:
        iou = _iou_xyxy(p.bbox, bbox)
        if iou > best_iou:
            best_iou = iou
            best_cat = cat
    _checked += 1
    if best_iou >= 0.3:
        patch_category[p.patch_id] = best_cat
        _matched += 1

match_rate = _matched / max(_checked, 1)
print(f"Kategori eslestirme (bbox IoU>=0.3): {_matched}/{_checked} object-patch (%{match_rate*100:.1f})")

if match_rate < 0.10:
    print("UYARI: eslesme orani cok dusuk -- bbox koordinat sistemleri UYUSMUYOR olabilir")
    print("(patch bboxlari ve COCO nesne bboxlari farkli olcekte olabilir).")
    print("Gercek-kategori hard-negative GUVENLI SEKILDE DEVRE DISI -- mevcut CV-tabanli")
    print("yontem hic degismeden kullanilmaya devam edecek.")
    REALNEG_ENABLED = False
else:
    print("Eslesme orani makul -- KONTROL 1 basarili.")
    REALNEG_ENABLED = True

# KONTROL 2: train_pairs'e gercekten yeterli oranda kategori-negatifi eklenebiliyor mu?
CATEGORY_HN_COVERAGE = 0.0
if REALNEG_ENABLED:
    img_id_lookup_r = {p.patch_id: p.image_id for p in hybrid_index.image_patches}
    category_to_train_images = {}
    for pid, cat in patch_category.items():
        iid = img_id_lookup_r[pid]
        if iid in train_image_ids:
            category_to_train_images.setdefault(cat, set()).add(iid)
    print(f"Train havuzunda {len(category_to_train_images)} farkli kategori temsil ediliyor.")

    by_image_patches_r = {}
    for p in hybrid_index.image_patches:
        if p.type == "object" and p.image_id in train_image_ids:
            by_image_patches_r.setdefault(p.image_id, []).append(p)

    emb_to_patch_r = {}
    for p in hybrid_index.image_patches:
        if p.image_id in train_image_ids:
            emb_to_patch_r[np.asarray(p.embedding, dtype=np.float32).tobytes()] = p

    _rng_cat = random.Random(777)
    _n_with_catneg = 0
    for pair in train_pairs:
        pair["category_hard_negative_embedding"] = None
        src_patch = emb_to_patch_r.get(pair["img_embedding"].tobytes())
        if src_patch is None:
            continue
        cat = patch_category.get(src_patch.patch_id)
        if cat is None:
            continue
        candidates = category_to_train_images.get(cat, set()) - {src_patch.image_id}
        if not candidates:
            continue
        other_img_id = _rng_cat.choice(sorted(candidates))
        other_patches = by_image_patches_r.get(other_img_id, [])
        if not other_patches:
            continue
        neg_patch = _rng_cat.choice(other_patches)
        pair["category_hard_negative_embedding"] = np.asarray(neg_patch.embedding, dtype=np.float32)
        _n_with_catneg += 1

    CATEGORY_HN_COVERAGE = _n_with_catneg / max(len(train_pairs), 1)
    print(f"Gercek-kategori hard-negative eklenen train cifti: {_n_with_catneg}/{len(train_pairs)} (%{CATEGORY_HN_COVERAGE*100:.1f})")

    if CATEGORY_HN_COVERAGE < 0.05:
        print("UYARI: nihai kapsam cok dusuk, anlamli bir egitim sinyali saglamayacak --")
        print("gercek-kategori hard-negative varyanti DEVRE DISI birakiliyor.")
        REALNEG_ENABLED = False
    else:
        print("KONTROL 2 basarili -- gercek-kategori hard-negative ETKIN, Bolum 19da")
        print("D_SACL_continuous_realneg_leakfree adiyla ek bir varyant olarak test edilecek.")


## 10. Zincir-Koruyan Veri (SACL için)

In [ ]:
# HÜCRE 8: Zincir yapısını (SACL bölümü için) koru ve kaydet

def build_chain_records(sequences, hybrid_index, min_chain_nodes=8):
    txt_lookup = {p.patch_id: p.embedding for p in hybrid_index.text_patches}
    img_lookup = {p.patch_id: p.embedding for p in hybrid_index.image_patches}
    chain_records = []
    for chain in sequences:
        nodes = []
        for step in chain:
            emb = img_lookup.get(step.patch_id) if step.modality == "image" else txt_lookup.get(step.patch_id)
            if emb is None:
                continue
            nodes.append({"step": step.step, "modality": 0 if step.modality == "image" else 1,
                           "embedding": np.asarray(emb, dtype=np.float32)})
        if len(nodes) >= min_chain_nodes:
            chain_records.append(nodes)
    return chain_records


if "train_sequences" in globals():
    train_chain_records = build_chain_records(train_sequences, hybrid_index, min_chain_nodes=8)
    with open(CHAIN_RECORDS_PATH, "wb") as f:
        pickle.dump(train_chain_records, f)
    avg_len = sum(len(c) for c in train_chain_records) / max(len(train_chain_records), 1)
    print(f"✅ {len(train_chain_records)} zincir korundu (ort. uzunluk {avg_len:.1f} adım) -> {CHAIN_RECORDS_PATH}")
elif os.path.exists(CHAIN_RECORDS_PATH):
    with open(CHAIN_RECORDS_PATH, "rb") as f:
        train_chain_records = pickle.load(f)
    print(f"⚠️ train_sequences bellekte yoktu, önceden kaydedilmiş dosyadan yüklendi: {len(train_chain_records)} zincir")
else:
    print("❌ train_sequences bellekte yok VE kaydedilmiş chain_records da yok.")
    print("   SACL bölümünü kullanmak istiyorsan Hücre 7'yi (train/test dosyaları")
    print("   SİLİNMİŞ halde) yeniden çalıştırıp train_sequences'i üretmen gerekiyor.")


## 11. Model Mimarileri (v4 ile aynı)

In [ ]:
# HÜCRE 10: Model mimarileri (v5h: residual/adapter mimari)
# Eskiden projeksiyon katmani ham embedding'i SIFIRDAN yeniden ogreniyordu
# (rastgele baslatilmis agirliklarla) VE 2048->512 boyut daraltmasi yapiyordu
# -- bu, kucuk/gurultulu bir egitim sinyaliyle zaten guclu olan ham Qwen
# temsilini bozma riski tasiyordu (nitekim bazi varyantlar ham modeli
# gecemiyordu). Simdi LoRA/adapter mantigiyla: agirliklar ham embedding'e
# KUCUK bir duzeltme (delta) ogreniyor, son katman SIFIR baslatiliyor --
# yani egitimin basinda delta=0 ve cikti TAM OLARAK ham (normalize edilmis)
# embedding'e esit. Egitim SADECE iyilestirme yonunde ilerleyebilir, ham
# modelden daha kotu bir noktadan baslamiyor.

EMBED_DIM = 2048
HIDDEN    = 1024
TEMP      = 0.07


class ResidualHead(nn.Module):
    """Ham embedding + kucuk ogrenilebilir duzeltme (delta). Son katman
    sifir-baslatilir -- baslangicta delta=0, cikti = normalize(ham_embedding)."""
    def __init__(self, embed_dim=EMBED_DIM, context_dim=0, hidden=HIDDEN):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(embed_dim + context_dim, hidden), nn.GELU(),
                                  nn.LayerNorm(hidden), nn.Linear(hidden, embed_dim))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, x, context=None):
        inp = x if context is None else torch.cat([x, context], dim=-1)
        return F.normalize(x + self.mlp(inp), dim=-1)


class ProjectionHead(nn.Module):
    """A) Baseline / B) Hard-negative-aware / E) SACL için ortak mimari."""
    def __init__(self, input_dim=EMBED_DIM, hidden=HIDDEN):
        super().__init__()
        self.img_proj = ResidualHead(input_dim, context_dim=0, hidden=hidden)
        self.txt_proj = ResidualHead(input_dim, context_dim=0, hidden=hidden)

    def forward(self, img, txt):
        return self.img_proj(img), self.txt_proj(txt)

    def project_image(self, img):
        return self.img_proj(img)


class InstructionAwareHead(nn.Module):
    """C) Instruction-aware (ORACLE mod gerektirir -- sadece üst-sınır/referans)."""
    def __init__(self, input_dim=EMBED_DIM, hidden=HIDDEN, num_modes=4):
        super().__init__()
        self.mode_emb = nn.Embedding(num_modes, input_dim)
        self.img_proj = ResidualHead(input_dim, context_dim=input_dim, hidden=hidden)
        self.txt_proj = ResidualHead(input_dim, context_dim=input_dim, hidden=hidden)

    def forward(self, img, txt, mode_idx):
        m = self.mode_emb(mode_idx)
        return self.img_proj(img, m), self.txt_proj(txt, m)

    def project_image(self, img, mode_idx):
        m = self.mode_emb(mode_idx)
        return self.img_proj(img, m)


class SelfInferringHead(nn.Module):
    """D) Self-inferring (ORACLE YOK) -- gerçek kullanım için asıl aday."""
    def __init__(self, input_dim=EMBED_DIM, hidden=HIDDEN, num_modes=4):
        super().__init__()
        self.mode_table = nn.Embedding(num_modes, input_dim)
        self.mode_predictor = nn.Sequential(nn.Linear(input_dim * 2, 256), nn.GELU(), nn.Linear(256, num_modes))
        self.img_proj = ResidualHead(input_dim, context_dim=input_dim, hidden=hidden)
        self.txt_proj = ResidualHead(input_dim, context_dim=input_dim, hidden=hidden)

    def infer_mode(self, img, txt):
        logits = self.mode_predictor(torch.cat([img, txt], dim=-1))
        probs = F.softmax(logits, dim=-1)
        mode_vec = probs @ self.mode_table.weight
        return mode_vec, logits

    def forward(self, img, txt):
        mode_vec, mode_logits = self.infer_mode(img, txt)
        z_img = self.img_proj(img, mode_vec)
        z_txt = self.txt_proj(txt, mode_vec)
        return z_img, z_txt, mode_logits

    def project_image_with_vec(self, img, mode_vec):
        return self.img_proj(img, mode_vec)


## 12. Loss Fonksiyonları (v4 ile aynı)

In [ ]:
# HÜCRE 11: Loss fonksiyonları

def plain_info_nce(img, txt, **kwargs):
    batch_size = img.size(0)
    logits = img @ txt.T / TEMP
    labels = torch.arange(batch_size, device=img.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2


def hn_info_nce(img, txt, z_hn, has_hn, **kwargs):
    batch_size = img.size(0)
    labels = torch.arange(batch_size, device=img.device)
    logits_i2t = img @ txt.T / TEMP
    loss_i2t = F.cross_entropy(logits_i2t, labels)
    logits_t2i_base = txt @ img.T / TEMP
    # DUZELTME: batch'te HIC hard-negative yoksa z_hn=None gelir -- eskiden
    # bu durumda dogrudan (txt * None) carpimina girip TypeError firlatiyordu.
    # Simdi z_hn None ise o ek sutunu atlayip normal InfoNCE'ye dusuyor.
    if z_hn is not None:
        sim_txt_hn = (txt * z_hn).sum(-1, keepdim=True) / TEMP
        sim_txt_hn = sim_txt_hn.masked_fill((has_hn.unsqueeze(1) == 0), float("-inf"))
        logits_t2i = torch.cat([logits_t2i_base, sim_txt_hn], dim=1)
    else:
        logits_t2i = logits_t2i_base
    loss_t2i = F.cross_entropy(logits_t2i, labels)
    return (loss_i2t + loss_t2i) / 2


def hn_info_nce_realneg(img, txt, z_hn, has_hn, z_chn=None, has_chn=None, **kwargs):
    """v5h: hn_info_nce'nin genisletilmisi -- CV-tabanli hard-negative'e (z_hn)
    EK olarak, gercek COCO kategori eslesmesiyle bulunmus semantik
    hard-negative'i (z_chn, varsa) de ayri bir sutun olarak ekliyor.
    z_chn/has_chn verilmezse davranis hn_info_nce ile BIREBIR AYNI."""
    batch_size = img.size(0)
    labels = torch.arange(batch_size, device=img.device)
    logits_i2t = img @ txt.T / TEMP
    loss_i2t = F.cross_entropy(logits_i2t, labels)
    logits_t2i_base = txt @ img.T / TEMP
    extra_cols = [logits_t2i_base]
    if z_hn is not None:
        sim_txt_hn = (txt * z_hn).sum(-1, keepdim=True) / TEMP
        sim_txt_hn = sim_txt_hn.masked_fill((has_hn.unsqueeze(1) == 0), float("-inf"))
        extra_cols.append(sim_txt_hn)
    if z_chn is not None and has_chn is not None:
        sim_txt_chn = (txt * z_chn).sum(-1, keepdim=True) / TEMP
        sim_txt_chn = sim_txt_chn.masked_fill((has_chn.unsqueeze(1) == 0), float("-inf"))
        extra_cols.append(sim_txt_chn)
    logits_t2i = torch.cat(extra_cols, dim=1)
    loss_t2i = F.cross_entropy(logits_t2i, labels)
    return (loss_i2t + loss_t2i) / 2


MODE_MAP = {"semantic": 0, "color": 1, "shape": 2, "texture": 3}
MODE_AUX_WEIGHT = 0.1  # tarama sonucu: 0.1 en iyi MRR (0.363), >=2.0 mod_aux baskinlasip R@1 dusuruyor


## 13. Dataset + Eğitim/Değerlendirme Döngüleri (v4 ile aynı)

In [ ]:
# HÜCRE 12: Dataset + eğitim/değerlendirme döngüleri

BATCH_SIZE = 128
LR = 1e-4


class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        img = torch.tensor(np.asarray(p["img_embedding"], dtype=np.float32))
        txt = torch.tensor(np.asarray(p["txt_embedding"], dtype=np.float32))
        hn = p.get("hard_negative_embedding")
        has_hn = hn is not None
        hn_t = torch.tensor(np.asarray(hn, dtype=np.float32)) if has_hn else torch.zeros_like(img)
        # Fikir 2: metin-tarafı hard-negative (varsa) -- yeniden üretilen veri kümesinde bulunur
        hn_txt = p.get("hard_negative_txt_embedding")
        has_hn_txt = hn_txt is not None
        hn_txt_t = torch.tensor(np.asarray(hn_txt, dtype=np.float32)) if has_hn_txt else torch.zeros_like(txt)
        mode = p.get("hard_negative_mode") or "semantic"
        mode_idx = MODE_MAP.get(mode, 0)
        # v5h: gercek COCO-kategori hard-negative (varsa) -- opsiyonel, geriye
        # donuk uyumlu (yoksa has_chn=0, mevcut yontemleri hic etkilemez).
        chn = p.get("category_hard_negative_embedding")
        has_chn = chn is not None
        chn_t = torch.tensor(np.asarray(chn, dtype=np.float32)) if has_chn else torch.zeros_like(img)
        return {"img": img, "txt": txt, "hn": hn_t,
                "has_hn": torch.tensor(1.0 if has_hn else 0.0),
                "hn_txt": hn_txt_t,
                "has_hn_txt": torch.tensor(1.0 if has_hn_txt else 0.0),
                "chn": chn_t,
                "has_chn": torch.tensor(1.0 if has_chn else 0.0),
                "mode_idx": torch.tensor(mode_idx, dtype=torch.long)}


train_loader = DataLoader(PairDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)


def train_model(model_type, loss_fn_name, epochs, val_pairs=None, patience=4, eval_every=5, loader=None, seed=None):
    """v5h: cosine LR schedule + val-tabanli erken durdurma + en-iyi-checkpoint.
    val_pairs verilirse, her eval_every epoch'ta bir val R@1 olculur; en iyi
    R@1'i veren agirliklar saklanir (deep copy) ve egitim sonunda GERI
    YUKLENIR -- yani donen model, son epoch'un degil, val'e gore EN IYI
    epoch'un modelidir. patience kontrol turunda iyilesme olmazsa erken durur.
    v5h-fix: 'seed' parametresi eklendi -- eskiden hep global SEED sabitine
    resetleniyordu, cok-seed dogrulamasindaki gercek seed varyasyonunu
    SESSIZCE eziyordu (A/B/C her seed'de birebir ayni sonucu veriyordu)."""
    torch.manual_seed(seed if seed is not None else SEED)
    if model_type == "plain":
        model = ProjectionHead().to(device)
    elif model_type == "instruction":
        model = InstructionAwareHead().to(device)
    else:
        model = SelfInferringHead().to(device)

    loss_fn = plain_info_nce if loss_fn_name == "plain" else hn_info_nce
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0
    checks_since_improve = 0

    data_loader = loader if loader is not None else train_loader

    for epoch in range(epochs):
        model.train()
        epoch_loss, epoch_aux = 0.0, 0.0
        for batch in data_loader:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            if model_type == "plain":
                z_img, z_txt = model(img, txt)
                z_hn = model.project_image(hn) if loss_fn_name == "hn" else None
                loss = loss_fn(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
                aux = torch.tensor(0.0)
            elif model_type == "instruction":
                z_img, z_txt = model(img, txt, mode_idx)
                z_hn = model.project_image(hn, mode_idx) if loss_fn_name == "hn" else None
                loss = loss_fn(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
                aux = torch.tensor(0.0)
            else:
                z_img, z_txt, mode_logits = model(img, txt)
                z_hn = None
                if loss_fn_name == "hn":
                    mode_vec_hn, _ = model.infer_mode(img, txt)
                    z_hn = model.project_image_with_vec(hn, mode_vec_hn)
                base_loss = loss_fn(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
                mask = has_hn.bool()
                aux = F.cross_entropy(mode_logits[mask], mode_idx[mask]) if mask.sum() > 0 else torch.tensor(0.0)
                loss = base_loss + MODE_AUX_WEIGHT * aux

            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); epoch_aux += aux.item()
        scheduler.step()

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  [{model_type}/{loss_fn_name}] epoch {epoch+1}/{epochs} loss={epoch_loss/len(data_loader):.4f} mode_aux={epoch_aux/len(data_loader):.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate(model, model_type, val_pairs, forced_mode=None)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]
                best_state = copy.deepcopy(model.state_dict())
                checks_since_improve = 0
            else:
                checks_since_improve += 1
            if checks_since_improve >= patience:
                print(f"  [erken durdurma] val R@1 {patience} kontroldur iyilesmedi (en iyi %{best_val_r1:.1f}), epoch {epoch+1}'de durduruluyor.")
                break

    if val_pairs is not None:
        model.load_state_dict(best_state)
        print(f"  [{model_type}/{loss_fn_name}] en iyi val R@1={best_val_r1:.1f}% olan checkpoint geri yuklendi.")
    return model


@torch.no_grad()
def evaluate(model, model_type, pairs, forced_mode=None):
    model.eval()
    img_all = torch.tensor(np.stack([np.asarray(p["img_embedding"], dtype=np.float32) for p in pairs])).to(device)
    txt_all = torch.tensor(np.stack([np.asarray(p["txt_embedding"], dtype=np.float32) for p in pairs])).to(device)

    if model_type == "plain":
        z_img, z_txt = model(img_all, txt_all)
    elif model_type == "instruction":
        if forced_mode is not None:
            mode_idx = torch.full((len(pairs),), MODE_MAP[forced_mode], dtype=torch.long, device=device)
        else:
            mode_idx = torch.tensor([MODE_MAP.get(p.get("hard_negative_mode") or "semantic", 0) for p in pairs], dtype=torch.long, device=device)
        z_img, z_txt = model(img_all, txt_all, mode_idx)
    else:
        z_img, z_txt, mode_logits = model(img_all, txt_all)
        pred_mode = mode_logits.argmax(dim=-1)
        true_mode = torch.tensor([MODE_MAP.get(p.get("hard_negative_mode") or "semantic", 0) for p in pairs], dtype=torch.long, device=device)
        acc = (pred_mode == true_mode).float().mean().item()
        print(f"  [self_infer] mod tahmin doğruluğu (bilgi amaçlı, oracle DEĞİL): %{acc*100:.1f}")

    sim = (z_txt @ z_img.T).cpu().numpy()
    n = len(pairs)
    r1 = r5 = r10 = 0
    mrr = 0.0
    for i in range(n):
        order = np.argsort(-sim[i])
        rank = int(np.where(order == i)[0][0]) + 1
        if rank == 1: r1 += 1
        if rank <= 5: r5 += 1
        if rank <= 10: r10 += 1
        mrr += 1.0 / rank
    return {"R@1": r1/n*100, "R@5": r5/n*100, "R@10": r10/n*100, "MRR": mrr/n}


## 14. SACL Çekirdek Fonksiyonlar (v4 ile aynı)

In [ ]:
# HÜCRE 14: SACL çekirdek sınıf/fonksiyonlar

class ChainBatch:
    def __init__(self, emb, modality, mask, step_idx):
        self.emb = emb
        self.modality = modality
        self.mask = mask
        self.step_idx = step_idx

    def to(self, device):
        return ChainBatch(self.emb.to(device), self.modality.to(device),
                           self.mask.to(device), self.step_idx.to(device))


def project_chain_batch(model, batch: "ChainBatch"):
    B, L, D = batch.emb.shape
    flat = batch.emb.view(B * L, D)
    z_img = F.normalize(model.img_proj(flat), dim=-1)
    z_txt = F.normalize(model.txt_proj(flat), dim=-1)
    mod = batch.modality.view(B * L, 1).float()
    z = z_img * (1 - mod) + z_txt * mod
    return z.view(B, L, -1)


def sequence_aware_loss(z, batch: "ChainBatch", temp=0.07, pos_window=4,
                         lambda_multipos=0.5, lambda_rank=0.5, margin=0.1,
                         max_rank_samples_per_chain=6, same_modality_only=False):
    """same_modality_only=False: ORİJİNAL SACL (v1) -- multipos/rank modaliteyi
    hesaba katmıyor, bir görsel düğümle pencere-içi bir metin düğümünü de
    "pozitif" sayıyor. LAMBDA_SEQ taramasında bu, en iyi ayarda bile
    A_baseline'ı geçemedi (bkz. sweep hücresi) -- muhtemel sebep: l_align zaten
    çapraz-modal eşleşmeyi net biçimde öğretiyor, multipos/rank'in modaliteyi
    görmezden gelmesi ona çelişen/gürültülü bir sinyal ekliyor.

    same_modality_only=True (SACL v2): multipos ve rank SADECE aynı modalitedeki
    düğümler arasında hesaplanır (görsel<->görsel, metin<->metin) -- "aynı
    içeriğin farklı zoom seviyeleri kümelenmeli" fikri, l_align'ın çapraz-modal
    hedefiyle çakışmadan, ona ek/tamamlayıcı bir intra-modal yapı öğretir.
    """
    B, L, Dp = z.shape
    device = z.device

    flat_z, flat_chain, flat_step, flat_mod = [], [], [], []
    for b in range(B):
        valid = batch.mask[b]
        n_valid = int(valid.sum().item())
        if n_valid == 0:
            continue
        flat_z.append(z[b, valid])
        flat_chain.append(torch.full((n_valid,), b, device=device, dtype=torch.long))
        flat_step.append(batch.step_idx[b, valid])
        flat_mod.append(batch.modality[b, valid])
    flat_z = torch.cat(flat_z, dim=0)
    flat_chain = torch.cat(flat_chain, dim=0)
    flat_step = torch.cat(flat_step, dim=0)
    flat_mod = torch.cat(flat_mod, dim=0)
    N = flat_z.size(0)

    sim = flat_z @ flat_z.T / temp
    same_chain = flat_chain.unsqueeze(0) == flat_chain.unsqueeze(1)
    same_modality = flat_mod.unsqueeze(0) == flat_mod.unsqueeze(1)
    step_dist = (flat_step.unsqueeze(0) - flat_step.unsqueeze(1)).abs()
    is_self = torch.eye(N, dtype=torch.bool, device=device)

    pos_mask = same_chain & (step_dist > 0) & (step_dist <= pos_window) & (~is_self)
    if same_modality_only:
        pos_mask = pos_mask & same_modality
    logits = sim.masked_fill(is_self, float("-inf"))
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    log_prob = log_prob.masked_fill(is_self, 0.0)
    pos_counts = pos_mask.sum(dim=1).clamp(min=1)
    per_anchor = -(log_prob * pos_mask).sum(dim=1) / pos_counts
    has_pos = pos_mask.sum(dim=1) > 0
    l_multipos = per_anchor[has_pos].mean() if has_pos.any() else torch.zeros((), device=device)

    rank_terms = []
    for b in range(B):
        idx_b = (flat_chain == b).nonzero(as_tuple=True)[0]
        if idx_b.numel() < 3:
            continue
        n_sample = min(max_rank_samples_per_chain, idx_b.numel())
        perm = idx_b[torch.randperm(idx_b.numel(), device=device)[:n_sample]]
        for a in perm:
            others = idx_b[idx_b != a]
            if same_modality_only:
                others = others[flat_mod[others] == flat_mod[a]]
            if others.numel() < 2:
                continue
            d = (flat_step[others] - flat_step[a]).abs()
            order = torch.argsort(d)
            near, far = others[order[0]], others[order[-1]]
            if d[order[-1]] <= d[order[0]]:
                continue
            s_near = (flat_z[a] * flat_z[near]).sum() / temp
            s_far = (flat_z[a] * flat_z[far]).sum() / temp
            rank_terms.append(F.relu(torch.tensor(margin, device=device) - (s_near - s_far)))
    l_rank = torch.stack(rank_terms).mean() if rank_terms else torch.zeros((), device=device)

    return lambda_multipos * l_multipos + lambda_rank * l_rank, {
        "l_multipos": l_multipos.item(), "l_rank": l_rank.item(),
        "n_pos_anchors": int(has_pos.sum().item()), "n_rank_triplets": len(rank_terms),
    }


def sequence_aware_loss_continuous(z, batch: "ChainBatch", temp=0.07, tau=3.0,
                                    lambda_multipos=0.5, lambda_rank=0.5, margin=0.1,
                                    max_rank_samples_per_chain=6, same_modality_only=True,
                                    max_weight_dist=None):
    """v5h: pos_window'daki SERT esik yerine, adim mesafesine gore YUMUSAK
    (exponentially decaying) agirlik kullanan surekli SACL varyanti --
    weight(i,j) = exp(-step_dist(i,j)/tau). Zincirdeki TUM ciftler
    kullanilir, yakin adimlar yuksek, uzak adimlar dusuk agirlikla katkida
    bulunur -- ikili (pozitif/degil) ayrim yerine surekli bir "kavramsal
    mesafe" sinyali. Rank (margin) kismi degismedi, sadece multipos kismi
    surekli agirliklandirildi."""
    B, L, Dp = z.shape
    device = z.device

    flat_z, flat_chain, flat_step, flat_mod = [], [], [], []
    for b in range(B):
        valid = batch.mask[b]
        n_valid = int(valid.sum().item())
        if n_valid == 0:
            continue
        flat_z.append(z[b, valid])
        flat_chain.append(torch.full((n_valid,), b, device=device, dtype=torch.long))
        flat_step.append(batch.step_idx[b, valid])
        flat_mod.append(batch.modality[b, valid])
    flat_z = torch.cat(flat_z, dim=0)
    flat_chain = torch.cat(flat_chain, dim=0)
    flat_step = torch.cat(flat_step, dim=0)
    flat_mod = torch.cat(flat_mod, dim=0)
    N = flat_z.size(0)

    sim = flat_z @ flat_z.T / temp
    same_chain = flat_chain.unsqueeze(0) == flat_chain.unsqueeze(1)
    same_modality = flat_mod.unsqueeze(0) == flat_mod.unsqueeze(1)
    step_dist = (flat_step.unsqueeze(0) - flat_step.unsqueeze(1)).abs().float()
    is_self = torch.eye(N, dtype=torch.bool, device=device)

    eligible = same_chain & (~is_self)
    if same_modality_only:
        eligible = eligible & same_modality
    weight = torch.exp(-step_dist / tau) * eligible.float()
    if max_weight_dist is not None:
        weight = weight * (step_dist <= max_weight_dist).float()

    logits = sim.masked_fill(is_self, float("-inf"))
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    log_prob = log_prob.masked_fill(is_self, 0.0)

    weight_sum = weight.sum(dim=1).clamp(min=1e-8)
    per_anchor = -(log_prob * weight).sum(dim=1) / weight_sum
    has_pos = weight.sum(dim=1) > 1e-6
    l_multipos = per_anchor[has_pos].mean() if has_pos.any() else torch.zeros((), device=device)

    rank_terms = []
    for b in range(B):
        idx_b = (flat_chain == b).nonzero(as_tuple=True)[0]
        if idx_b.numel() < 3:
            continue
        n_sample = min(max_rank_samples_per_chain, idx_b.numel())
        perm = idx_b[torch.randperm(idx_b.numel(), device=device)[:n_sample]]
        for a in perm:
            others = idx_b[idx_b != a]
            if same_modality_only:
                others = others[flat_mod[others] == flat_mod[a]]
            if others.numel() < 2:
                continue
            d = (flat_step[others] - flat_step[a]).abs()
            order = torch.argsort(d)
            near, far = others[order[0]], others[order[-1]]
            if d[order[-1]] <= d[order[0]]:
                continue
            s_near = (flat_z[a] * flat_z[near]).sum() / temp
            s_far = (flat_z[a] * flat_z[far]).sum() / temp
            rank_terms.append(F.relu(torch.tensor(margin, device=device) - (s_near - s_far)))
    l_rank = torch.stack(rank_terms).mean() if rank_terms else torch.zeros((), device=device)

    return lambda_multipos * l_multipos + lambda_rank * l_rank, {
        "l_multipos": l_multipos.item(), "l_rank": l_rank.item(),
        "n_pos_anchors": int(has_pos.sum().item()), "n_rank_triplets": len(rank_terms),
    }


## 15. SACL Altyapısı (ChainDataset + proj. fonksiyonu -- koşu Bölüm 20/21'de)

Burada sadece SACL için gereken zincir veri altyapısını (`ChainDataset`,
`chain_loader`, `project_chain_batch_self_infer`) kuruyoruz. Gerçek
D + SACL-lite EĞİTİMİ ve DEĞERLENDİRMESİ, sızıntısız (leak-free) mimariyle
Bölüm 20'de yapılıyor -- burada eski/sızıntılı bir koşu YAPMIYORUZ.

In [ ]:
class ChainDataset(Dataset):
    def __init__(self, chain_records, max_len=65, embed_dim=EMBED_DIM):
        self.chains = chain_records
        self.max_len = max_len
        self.embed_dim = embed_dim

    def __len__(self):
        return len(self.chains)

    def __getitem__(self, idx):
        nodes = self.chains[idx][:self.max_len]
        emb = torch.zeros(self.max_len, self.embed_dim)
        modality = torch.zeros(self.max_len, dtype=torch.long)
        mask = torch.zeros(self.max_len, dtype=torch.bool)
        step_idx = torch.zeros(self.max_len, dtype=torch.long)
        for i, n in enumerate(nodes):
            emb[i] = torch.tensor(n["embedding"]); modality[i] = n["modality"]
            step_idx[i] = n["step"]; mask[i] = True
        return emb, modality, mask, step_idx


def chain_collate(batch_list):
    emb = torch.stack([b[0] for b in batch_list])
    modality = torch.stack([b[1] for b in batch_list])
    mask = torch.stack([b[2] for b in batch_list])
    step_idx = torch.stack([b[3] for b in batch_list])
    return ChainBatch(emb, modality, mask, step_idx)


CHAIN_BATCH_SIZE = 16
POS_WINDOW = 4
LAMBDA_MULTIPOS = 0.5
LAMBDA_RANK = 0.5
RANK_MARGIN = 0.1
SACL_MAX_LEN = 65

chain_loader = DataLoader(ChainDataset(train_chain_records, max_len=SACL_MAX_LEN),
                           batch_size=CHAIN_BATCH_SIZE, shuffle=True, drop_last=True,
                           collate_fn=chain_collate)


In [ ]:
def project_chain_batch_self_infer(model, batch: "ChainBatch"):
    """SelfInferringHead için zincir-düğümü projeksiyonu -- v4 ile aynı."""
    B, L, D = batch.emb.shape
    flat = batch.emb.view(B * L, D)
    mode_vec, _ = model.infer_mode(flat, flat)
    z_img = model.img_proj(flat, mode_vec)
    z_txt = model.txt_proj(flat, mode_vec)
    mod = batch.modality.view(B * L, 1).float()
    z = z_img * (1 - mod) + z_txt * mod
    return z.view(B, L, -1)


## 16. Train/Test Leakage Kontrolü (paper öncesi ZORUNLU)

v4'te test kümesinin %93.8'inin train ile birebir örtüştüğü bulundu. Aynı
kontrolü burada v5 için yapıyoruz -- bu satır boş dönmeden (ya da çok düşük
bir oranla dönmeden) hiçbir v5 sonucu paper'a konulmamalı.

In [ ]:
# HÜCRE L1: v5 train/val/test embedding-duzeyinde ortusme kontrolu (UCU de)
DRIVE_BASE = "/content/drive/MyDrive/COSMOS"

def emb_hash(p):
    return (p["img_embedding"].tobytes(), p["txt_embedding"].tobytes())

def img_hash(p):
    return p["img_embedding"].tobytes()

def check_overlap(name_a, pairs_a, name_b, pairs_b):
    hashes_a = set(emb_hash(p) for p in pairs_a)
    hashes_b = [emb_hash(p) for p in pairs_b]
    overlap = sum(1 for h in hashes_b if h in hashes_a)
    img_hashes_a = set(img_hash(p) for p in pairs_a)
    img_overlap = sum(1 for p in pairs_b if img_hash(p) in img_hashes_a)
    print(f"{name_b}/{name_a} TAM (gorsel+metin) ortusme: {overlap} / {len(pairs_b)} (%{overlap/len(pairs_b)*100:.1f})")
    print(f"{name_b}/{name_a} SADECE gorsel ortusme:      {img_overlap} / {len(pairs_b)} (%{img_overlap/len(pairs_b)*100:.1f})")
    return overlap, img_overlap

print("=== TRAIN vs TEST ===")
overlap_tt, img_overlap_tt = check_overlap("train", train_pairs, "test", test_pairs)
print("\n=== TRAIN vs VAL ===")
overlap_tv, img_overlap_tv = check_overlap("train", train_pairs, "val", val_pairs)
print("\n=== VAL vs TEST ===")
overlap_vt, img_overlap_vt = check_overlap("val", val_pairs, "test", test_pairs)

dup_in_test = len([img_hash(p) for p in test_pairs]) - len(set(img_hash(p) for p in test_pairs))
dup_in_val = len([img_hash(p) for p in val_pairs]) - len(set(img_hash(p) for p in val_pairs))
print(f"\nTest icinde duplike gorsel: {dup_in_test} | Val icinde duplike gorsel: {dup_in_val}")

LEAKAGE_REPORT = {
    "train_test_exact": overlap_tt, "train_test_img": img_overlap_tt,
    "train_val_exact": overlap_tv, "train_val_img": img_overlap_tv,
    "val_test_exact": overlap_vt, "val_test_img": img_overlap_vt,
    "dup_in_test": dup_in_test, "dup_in_val": dup_in_val,
}
import pickle, os
os.makedirs(f"{DRIVE_BASE}/ZoomInOut/v5i_results", exist_ok=True)
with open(f"{DRIVE_BASE}/ZoomInOut/v5i_results/leakage_report.pkl", "wb") as f:
    pickle.dump(LEAKAGE_REPORT, f)
print("\nKaydedildi -> v5i_results/leakage_report.pkl")

worst_pct = max(overlap_tt/len(test_pairs), overlap_tv/len(val_pairs), overlap_vt/len(test_pairs))
if worst_pct > 0.05:
    print("\n*** UYARI: bir ciftte anlamli duzeyde leakage var, sonuclar supheli! ***")
else:
    print("\n*** train/val/test ayrimi TEMIZ (ucu de birbirinden ayrik). ***")


## 17. Veri Kümesi İstatistikleri (paper Tablo 1 için)

In [ ]:
# HÜCRE L2: v5 veri kumesi istatistik tablosu
import numpy as np

n_unique_train_img = len(set(p["img_embedding"].tobytes() for p in train_pairs))
n_unique_val_img = len(set(p["img_embedding"].tobytes() for p in val_pairs))
n_unique_test_img = len(set(p["img_embedding"].tobytes() for p in test_pairs))
n_with_hn = sum(1 for p in train_pairs if p.get("hard_negative_embedding") is not None)
chain_lens = [len(c) for c in train_chain_records]

DATASET_STATS = {
    "train_pairs": len(train_pairs), "val_pairs": len(val_pairs), "test_pairs": len(test_pairs),
    "unique_train_images": n_unique_train_img, "unique_val_images": n_unique_val_img, "unique_test_images": n_unique_test_img,
    "hard_negative_coverage_pct": n_with_hn / len(train_pairs) * 100,
    "n_chains": len(train_chain_records),
    "avg_chain_len": float(np.mean(chain_lens)), "min_chain_len": int(np.min(chain_lens)), "max_chain_len": int(np.max(chain_lens)),
    "coco_caption_used": bool(COCO_MAPPING_READY),
}
print("=== v5 Veri Kümesi İstatistikleri ===")
for k, v in DATASET_STATS.items():
    print(f"  {k}: {v}")

with open(f"{DRIVE_BASE}/ZoomInOut/v5i_results/dataset_stats.pkl", "wb") as f:
    pickle.dump(DATASET_STATS, f)
print("\nKaydedildi -> v5i_results/dataset_stats.pkl")


## 17b. Kalibrasyon Referansı: Rastgele Şans + Ham Qwen (eğitimsiz)

Literatürdeki büyük ölçekli SOTA sayılarıyla (R@1 %50-80) doğrudan kıyaslanamayız
(ölçek çok farklı). Bunun yerine KENDİ test kümemizin doğal zorluk tavanını
ölçüyoruz: (1) rastgele şans (matematiksel taban), (2) ham/eğitimsiz Qwen
embedding'i (bu test kümesi, hiç eğitim yapmadan doğası gereği ne kadar
"kolay/zor"). Eğitilmiş modelleri, literatür yerine BU iki referansa göre
değerlendireceğiz -- gerçek katkı = eğitilmiş sonuç - ham sonuç.

In [ ]:
# HÜCRE L2b: kalibrasyon referanslari (v5e test kumesi uzerinde)
import numpy as np

@torch.no_grad()
def evaluate_raw_embeddings(pairs):
    img_all = torch.tensor(np.stack([np.asarray(p["img_embedding"], dtype=np.float32) for p in pairs])).to(device)
    txt_all = torch.tensor(np.stack([np.asarray(p["txt_embedding"], dtype=np.float32) for p in pairs])).to(device)
    z_img = F.normalize(img_all, dim=-1)
    z_txt = F.normalize(txt_all, dim=-1)
    sim = (z_txt @ z_img.T).cpu().numpy()
    n = len(pairs)
    r1 = r5 = r10 = 0
    mrr = 0.0
    for i in range(n):
        order = np.argsort(-sim[i])
        rank = int(np.where(order == i)[0][0]) + 1
        if rank == 1: r1 += 1
        if rank <= 5: r5 += 1
        if rank <= 10: r10 += 1
        mrr += 1.0 / rank
    return {"R@1": r1/n*100, "R@5": r5/n*100, "R@10": r10/n*100, "MRR": mrr/n}


def random_chance_baseline(pairs, n_trials=20):
    n = len(pairs)
    rng = np.random.RandomState(0)
    r1s, r5s, r10s, mrrs = [], [], [], []
    for _ in range(n_trials):
        r1 = r5 = r10 = 0
        mrr = 0.0
        for i in range(n):
            order = rng.permutation(n)
            rank = int(np.where(order == i)[0][0]) + 1
            if rank == 1: r1 += 1
            if rank <= 5: r5 += 1
            if rank <= 10: r10 += 1
            mrr += 1.0 / rank
        r1s.append(r1/n*100); r5s.append(r5/n*100); r10s.append(r10/n*100); mrrs.append(mrr/n)
    return {"R@1": np.mean(r1s), "R@5": np.mean(r5s), "R@10": np.mean(r10s), "MRR": np.mean(mrrs)}


r_random = random_chance_baseline(test_pairs)
r_ham = evaluate_raw_embeddings(test_pairs)

print("=== KALİBRASYON REFERANSLARI (v5e test kümesi, N={}) ===".format(len(test_pairs)))
print(f"{'Referans':<28} | {'R@1':>7} | {'R@5':>7} | {'R@10':>7} | {'MRR':>7}")
print("-" * 70)
print(f"{'Rastgele şans (taban)':<28} | {r_random['R@1']:>6.1f}% | {r_random['R@5']:>6.1f}% | {r_random['R@10']:>6.1f}% | {r_random['MRR']:.3f}")
print(f"{'Ham Qwen (eğitimsiz)':<28} | {r_ham['R@1']:>6.1f}% | {r_ham['R@5']:>6.1f}% | {r_ham['R@10']:>6.1f}% | {r_ham['MRR']:.3f}")
print("-" * 70)
print("Not: Eğitilmiş modelleri (Bölüm 20-22) bu iki referansa göre değerlendir --")
print("     literatürdeki büyük-ölçek SOTA sayılarıyla DOĞRUDAN kıyaslanamaz.")

CALIBRATION_REFERENCE = {"random_chance": r_random, "ham_qwen": r_ham, "n_test": len(test_pairs)}
with open(f"{DRIVE_BASE}/ZoomInOut/v5i_results/calibration_reference.pkl", "wb") as f:
    pickle.dump(CALIBRATION_REFERENCE, f)
print("\nKaydedildi -> v5i_results/calibration_reference.pkl")


## 18. Leak-Free D / D + SACL-lite Eğitimi (doğru mimari)

D'nin ilk değerlendirme yaklaşımı (`evaluate_leak_free` adıyla denenen ilk
düzeltme) YANLIŞTI: modeli çapraz-eşlenmiş (`infer_mode(img,txt)`) eğitip,
sadece test sırasında kendine-eşlenmiş (`infer_mode(img,img)`)
değerlendirmek dağılım dışı (out-of-distribution) bir durum yaratıyor.
Doğru düzeltme: modeli EĞİTİM sırasında da tutarlı biçimde kendine-eşlenmiş
mod çıkarımıyla eğitmek (aşağıdaki `train_model_D_leakfree` /
`train_d_sacl_lite_leakfree`).

In [ ]:
# HÜCRE L3: leak-free D ve D+SACL-lite egitim fonksiyonlari

@torch.no_grad()
def evaluate_self_paired(model, pairs):
    """D / D+SACL-lite icin dogru, sizintisiz degerlendirme -- egitimle TUTARLI
    (kendine-eslenmis mod cikarimi hem egitimde hem testte kullanilir)."""
    model.eval()
    img_all = torch.tensor(np.stack([np.asarray(p["img_embedding"], dtype=np.float32) for p in pairs])).to(device)
    txt_all = torch.tensor(np.stack([np.asarray(p["txt_embedding"], dtype=np.float32) for p in pairs])).to(device)
    mode_vec_img, _ = model.infer_mode(img_all, img_all)
    mode_vec_txt, _ = model.infer_mode(txt_all, txt_all)
    z_img = model.img_proj(img_all, mode_vec_img)
    z_txt = model.txt_proj(txt_all, mode_vec_txt)
    sim = (z_txt @ z_img.T).cpu().numpy()
    n = len(pairs)
    r1 = r5 = r10 = 0
    mrr = 0.0
    for i in range(n):
        order = np.argsort(-sim[i])
        rank = int(np.where(order == i)[0][0]) + 1
        if rank == 1: r1 += 1
        if rank <= 5: r5 += 1
        if rank <= 10: r10 += 1
        mrr += 1.0 / rank
    return {"R@1": r1/n*100, "R@5": r5/n*100, "R@10": r10/n*100, "MRR": mrr/n}


def train_model_D_leakfree(epochs=75, seed=None, val_pairs=None, patience=4, eval_every=5):
    torch.manual_seed(seed if seed is not None else SEED)
    model = SelfInferringHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0
    checks_since_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = epoch_aux = 0.0
        for batch in train_loader:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            # kendine-eslenmis mod cikarimi -- egitimde de TUTARLI
            mode_vec_img, mode_logits = model.infer_mode(img, img)
            mode_vec_txt, _ = model.infer_mode(txt, txt)
            z_img = model.img_proj(img, mode_vec_img)
            z_txt = model.txt_proj(txt, mode_vec_txt)
            z_hn = model.project_image_with_vec(hn, mode_vec_img) if has_hn.sum() > 0 else None

            base_loss = hn_info_nce(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
            mask = has_hn.bool()
            aux = F.cross_entropy(mode_logits[mask], mode_idx[mask]) if mask.sum() > 0 else torch.tensor(0.0, device=device)
            loss = base_loss + MODE_AUX_WEIGHT * aux
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); epoch_aux += aux.item()
        scheduler.step()
        if (epoch + 1) % 25 == 0 or epoch == 0:
            n = len(train_loader)
            print(f"  [D-leakfree] epoch {epoch+1}/{epochs} loss={epoch_loss/n:.4f} mode_aux={epoch_aux/n:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate_self_paired(model, val_pairs)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]
                best_state = copy.deepcopy(model.state_dict())
                checks_since_improve = 0
            else:
                checks_since_improve += 1
            if checks_since_improve >= patience:
                print(f"  [D-leakfree erken durdurma] val R@1 {patience} kontroldur iyilesmedi (en iyi %{best_val_r1:.1f}), epoch {epoch+1}'de durduruluyor.")
                break

    if val_pairs is not None:
        model.load_state_dict(best_state)
        print(f"  [D-leakfree] en iyi val R@1={best_val_r1:.1f}% olan checkpoint geri yuklendi.")
    return model


def train_d_sacl_lite_leakfree(epochs=75, lambda_seq=0.1, pos_window=2, seed=None, val_pairs=None, patience=4, eval_every=5):
    torch.manual_seed(seed if seed is not None else SEED)
    model = SelfInferringHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    chain_iter = itertools.cycle(chain_loader)
    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0
    checks_since_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = epoch_aux = epoch_seq = 0.0
        for batch in train_loader:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            mode_vec_img, mode_logits = model.infer_mode(img, img)
            mode_vec_txt, _ = model.infer_mode(txt, txt)
            z_img = model.img_proj(img, mode_vec_img)
            z_txt = model.txt_proj(txt, mode_vec_txt)
            z_hn = model.project_image_with_vec(hn, mode_vec_img) if has_hn.sum() > 0 else None

            base_loss = hn_info_nce(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
            mask = has_hn.bool()
            aux = F.cross_entropy(mode_logits[mask], mode_idx[mask]) if mask.sum() > 0 else torch.tensor(0.0, device=device)

            cb = next(chain_iter).to(device)
            z_chain = project_chain_batch_self_infer(model, cb)
            l_seq, _ = sequence_aware_loss(z_chain, cb, pos_window=pos_window,
                                            lambda_multipos=LAMBDA_MULTIPOS, lambda_rank=LAMBDA_RANK,
                                            margin=RANK_MARGIN, same_modality_only=True)

            loss = base_loss + MODE_AUX_WEIGHT * aux + lambda_seq * l_seq
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); epoch_aux += aux.item(); epoch_seq += l_seq.item()
        scheduler.step()
        if (epoch + 1) % 25 == 0 or epoch == 0:
            n = len(train_loader)
            print(f"  [D+SACL-lite-leakfree] epoch {epoch+1}/{epochs} loss={epoch_loss/n:.4f} mode_aux={epoch_aux/n:.4f} seq={epoch_seq/n:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate_self_paired(model, val_pairs)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]
                best_state = copy.deepcopy(model.state_dict())
                checks_since_improve = 0
            else:
                checks_since_improve += 1
            if checks_since_improve >= patience:
                print(f"  [D+SACL-lite erken durdurma] val R@1 {patience} kontroldur iyilesmedi (en iyi %{best_val_r1:.1f}), epoch {epoch+1}'de durduruluyor.")
                break

    if val_pairs is not None:
        model.load_state_dict(best_state)
        print(f"  [D+SACL-lite-leakfree] en iyi val R@1={best_val_r1:.1f}% olan checkpoint geri yuklendi.")
    return model


def train_d_sacl_continuous_leakfree(epochs=75, lambda_seq=0.1, tau=5.0, seed=None, val_pairs=None, patience=4, eval_every=5):
    """v5h: D+SACL-lite'in surekli/mesafe-agirlikli (pos_window YERINE tau)
    varyanti -- sequence_aware_loss_continuous kullanir."""
    torch.manual_seed(seed if seed is not None else SEED)
    model = SelfInferringHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    chain_iter = itertools.cycle(chain_loader)
    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0
    checks_since_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = epoch_aux = epoch_seq = 0.0
        for batch in train_loader:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            mode_vec_img, mode_logits = model.infer_mode(img, img)
            mode_vec_txt, _ = model.infer_mode(txt, txt)
            z_img = model.img_proj(img, mode_vec_img)
            z_txt = model.txt_proj(txt, mode_vec_txt)
            z_hn = model.project_image_with_vec(hn, mode_vec_img) if has_hn.sum() > 0 else None

            base_loss = hn_info_nce(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
            mask = has_hn.bool()
            aux = F.cross_entropy(mode_logits[mask], mode_idx[mask]) if mask.sum() > 0 else torch.tensor(0.0, device=device)

            cb = next(chain_iter).to(device)
            z_chain = project_chain_batch_self_infer(model, cb)
            l_seq, _ = sequence_aware_loss_continuous(z_chain, cb, tau=tau,
                                            lambda_multipos=LAMBDA_MULTIPOS, lambda_rank=LAMBDA_RANK,
                                            margin=RANK_MARGIN, same_modality_only=True)

            loss = base_loss + MODE_AUX_WEIGHT * aux + lambda_seq * l_seq
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); epoch_aux += aux.item(); epoch_seq += l_seq.item()
        scheduler.step()
        if (epoch + 1) % 25 == 0 or epoch == 0:
            n = len(train_loader)
            print(f"  [D+SACL-continuous] epoch {epoch+1}/{epochs} loss={epoch_loss/n:.4f} mode_aux={epoch_aux/n:.4f} seq={epoch_seq/n:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate_self_paired(model, val_pairs)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]
                best_state = copy.deepcopy(model.state_dict())
                checks_since_improve = 0
            else:
                checks_since_improve += 1
            if checks_since_improve >= patience:
                print(f"  [D+SACL-continuous erken durdurma] val R@1 {patience} kontroldur iyilesmedi (en iyi %{best_val_r1:.1f}), epoch {epoch+1}'de durduruluyor.")
                break

    if val_pairs is not None:
        model.load_state_dict(best_state)
        print(f"  [D+SACL-continuous] en iyi val R@1={best_val_r1:.1f}% olan checkpoint geri yuklendi.")
    return model


def train_d_sacl_continuous_realneg_leakfree(epochs=75, lambda_seq=0.1, tau=5.0, seed=None, val_pairs=None, patience=4, eval_every=5):
    """v5h: D+SACL-continuous + gercek COCO-kategori hard-negative (varsa,
    REALNEG_ENABLED=True ise). PairDataset zaten chn/has_chn dondurdugu
    icin, tek fark hn_info_nce_realneg kullanmak."""
    torch.manual_seed(seed if seed is not None else SEED)
    model = SelfInferringHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    chain_iter = itertools.cycle(chain_loader)
    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0
    checks_since_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = epoch_aux = epoch_seq = 0.0
        for batch in train_loader:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            chn = batch["chn"].to(device); has_chn = batch["has_chn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            mode_vec_img, mode_logits = model.infer_mode(img, img)
            mode_vec_txt, _ = model.infer_mode(txt, txt)
            z_img = model.img_proj(img, mode_vec_img)
            z_txt = model.txt_proj(txt, mode_vec_txt)
            z_hn = model.project_image_with_vec(hn, mode_vec_img) if has_hn.sum() > 0 else None
            z_chn = model.project_image_with_vec(chn, mode_vec_img) if has_chn.sum() > 0 else None

            base_loss = hn_info_nce_realneg(z_img, z_txt, z_hn=z_hn, has_hn=has_hn, z_chn=z_chn, has_chn=has_chn)
            mask = has_hn.bool()
            aux = F.cross_entropy(mode_logits[mask], mode_idx[mask]) if mask.sum() > 0 else torch.tensor(0.0, device=device)

            cb = next(chain_iter).to(device)
            z_chain = project_chain_batch_self_infer(model, cb)
            l_seq, _ = sequence_aware_loss_continuous(z_chain, cb, tau=tau,
                                            lambda_multipos=LAMBDA_MULTIPOS, lambda_rank=LAMBDA_RANK,
                                            margin=RANK_MARGIN, same_modality_only=True)

            loss = base_loss + MODE_AUX_WEIGHT * aux + lambda_seq * l_seq
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); epoch_aux += aux.item(); epoch_seq += l_seq.item()
        scheduler.step()
        if (epoch + 1) % 25 == 0 or epoch == 0:
            n = len(train_loader)
            print(f"  [D+SACL-continuous-realneg] epoch {epoch+1}/{epochs} loss={epoch_loss/n:.4f} mode_aux={epoch_aux/n:.4f} seq={epoch_seq/n:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate_self_paired(model, val_pairs)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]
                best_state = copy.deepcopy(model.state_dict())
                checks_since_improve = 0
            else:
                checks_since_improve += 1
            if checks_since_improve >= patience:
                print(f"  [D+SACL-continuous-realneg erken durdurma] val R@1 {patience} kontroldur iyilesmedi (en iyi %{best_val_r1:.1f}), epoch {epoch+1}'de durduruluyor.")
                break

    if val_pairs is not None:
        model.load_state_dict(best_state)
        print(f"  [D+SACL-continuous-realneg] en iyi val R@1={best_val_r1:.1f}% olan checkpoint geri yuklendi.")
    return model


SINGLE_RUN_MARKER = f"{DRIVE_BASE}/ZoomInOut/v5i_results/single_run_models.pkl"

if os.path.exists(SINGLE_RUN_MARKER):
    print("=== Tek koşu doğrulaması zaten yapılmıştı, ATLANIYOR ===")
    print("    (mimari doğrulaması daha önce yapıldı -- yeniden eğitmeye gerek yok,")
    print("     asıl istatistiksel sonuç zaten Bölüm 19'daki çok-seed'den geliyor)")
    with open(SINGLE_RUN_MARKER, "rb") as f:
        _single_run_cache = pickle.load(f)
    r_d_test, r_dsacl_test, r_dsacl_cont_test = _single_run_cache["r_d"], _single_run_cache["r_dsacl"], _single_run_cache["r_dsacl_cont"]
    print(f"D (leak-free) -> R@1={r_d_test['R@1']:.1f}%  MRR={r_d_test['MRR']:.3f}")
    print(f"D+SACL-lite (leak-free) -> R@1={r_dsacl_test['R@1']:.1f}%  MRR={r_dsacl_test['MRR']:.3f}")
    print(f"D+SACL-continuous (leak-free) -> R@1={r_dsacl_cont_test['R@1']:.1f}%  MRR={r_dsacl_cont_test['MRR']:.3f}")
    print("⚠️  NOT: _m_dsacl_test bu yoldan YÜKLENEMEZ (sadece metrikler kaydedildi).")
    print("    Bölüm 21 (nitel örnekler) de kendi önbelleğinden atlayacak, sorun olmaz.")
else:
    print("=== Tek koşu doğrulaması (leak-free mimari doğru mu diye) ===")
    _m_d_test = train_model_D_leakfree(epochs=75, seed=SEED, val_pairs=val_pairs)
    r_d_test = evaluate_self_paired(_m_d_test, test_pairs)
    print(f"D (leak-free) -> R@1={r_d_test['R@1']:.1f}%  MRR={r_d_test['MRR']:.3f}")

    _m_dsacl_test = train_d_sacl_lite_leakfree(epochs=75, lambda_seq=0.1, pos_window=2, seed=SEED, val_pairs=val_pairs)
    r_dsacl_test = evaluate_self_paired(_m_dsacl_test, test_pairs)
    print(f"D+SACL-lite (leak-free) -> R@1={r_dsacl_test['R@1']:.1f}%  MRR={r_dsacl_test['MRR']:.3f}")

    _m_dsacl_cont_test = train_d_sacl_continuous_leakfree(epochs=75, lambda_seq=0.1, tau=5.0, seed=SEED, val_pairs=val_pairs)
    r_dsacl_cont_test = evaluate_self_paired(_m_dsacl_cont_test, test_pairs)
    print(f"D+SACL-continuous (leak-free) -> R@1={r_dsacl_cont_test['R@1']:.1f}%  MRR={r_dsacl_cont_test['MRR']:.3f}")

    with open(SINGLE_RUN_MARKER, "wb") as f:
        pickle.dump({"r_d": r_d_test, "r_dsacl": r_dsacl_test, "r_dsacl_cont": r_dsacl_cont_test}, f)


## 18b. Vanilla-Random-Pairs Kontrol Kümesi (paper için kritik kontrol grubu)

Zoom-in-out motoru HİÇ kullanılmadan, train havuzundaki görsellerden RASTGELE
(görsel, COCO caption) çiftleri seçiyoruz -- hiçbir traversal, hiçbir
hard-negative mantığı yok. Bunu A_baseline ile AYNI mimari/kayıpla eğitip
karşılaştıracağız: eğer A_baseline (motor tarafından seçilmiş çiftler) bunu
geçemezse, motorun veri seçiminin (SACL'dan bağımsız olarak) hiçbir katkısı
yok demektir -- tüm katkı sadece sekans kaybından geliyor. Geçerse, motorun
kendi veri seçimi bile değerli bir sinyal taşıyor demektir.

In [ ]:
# HÜCRE L3b: vanilla-random-pairs kontrol kumesi uretimi
def generate_random_pairs(hybrid_index, image_ids, n_samples, seed=99):
    """v5h: VANILLA KONTROL -- zoom motoru hic kullanilmadan, train
    havuzundaki gorsellerden rastgele (gorsel, COCO caption) ciftleri
    secer. Hard-negative alanlari hep None (bu kontrol grubunda anlamsiz)."""
    rng = random.Random(seed)
    # DETERMINIZM: train_image_ids bir SET ve Python string hash'ini her
    # surecte rastgele tohumluyor -> list(set) sirasi her Colab oturumunda
    # degisiyordu. rng=Random(99) sabit olsa bile FARKLI SIRALI listeye
    # indeksledigi icin her oturum BASKA bir cift kumesi uretiliyordu.
    image_ids = sorted(image_ids)
    img_id_lookup = {p.patch_id: p.image_id for p in hybrid_index.image_patches}
    by_image = {}
    for p in hybrid_index.image_patches:
        by_image.setdefault(p.image_id, []).append(p)

    pairs = []
    attempts, max_attempts = 0, n_samples * 30
    while len(pairs) < n_samples and attempts < max_attempts:
        attempts += 1
        img_id = rng.choice(image_ids)
        coco_id = patch_image_id_to_coco_id.get(img_id) if COCO_MAPPING_READY else None
        if coco_id is None or coco_id not in coco_caption_embeddings:
            continue
        pool = by_image.get(img_id, [])
        if not pool:
            continue
        obj_pool = [p for p in pool if p.type == "object"]
        patch = rng.choice(obj_pool if obj_pool else pool)
        _cap_text, cap_emb = rng.choice(coco_caption_embeddings[coco_id])
        pairs.append({
            "img_embedding": np.asarray(patch.embedding, dtype=np.float32),
            "txt_embedding": np.asarray(cap_emb, dtype=np.float32),
            "cross_modal_sim": None,
            "hard_negative_embedding": None, "hard_negative_sim": None, "hard_negative_mode": None,
            "hard_negative_txt_embedding": None, "hard_negative_txt_sim": None,
        })
    return pairs


random_pairs = generate_random_pairs(hybrid_index, train_image_ids, len(train_pairs), seed=99)
print(f"Vanilla-random kontrol kumesi: {len(random_pairs)} cift uretildi (hedef: {len(train_pairs)})")
random_loader = DataLoader(PairDataset(random_pairs), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)


## 19. Çok-Seed Doğrulama (paper için ana tablo -- EN AĞIR BÖLÜM)

A/B/C/D/D+SACL-lite'ı 5 farklı seed ile yeniden eğitip ortalama±std
raporluyoruz. Her seed'den SONRA sonuç Drive'a kaydediliyor -- Colab
bağlantısı kopsa bile, bu hücreyi tekrar çalıştırdığında `seed_results`
dosyadan geri yüklenip KALDIĞI SEED'DEN devam eder (baştan başlamaz).

In [ ]:
# HÜCRE GPU-KAYIT: donanımı kaydet ve TF32 durumunu sabitle
# L4-TIMED'DAN ÖNCE çalıştırılır. Eğitim yok, saniyeler sürer.
# ---------------------------------------------------------------------------
# NEDEN: sure kolonu bildirildigi icin makalenin donanimi adiyla soylemesi
# gerekiyor; notebook bunu hicbir yerde kaydetmiyordu. Bu hucre donanimi
# donanim.json'a yazar, sonraki oturumlarda ayni GPU'da olup olmadigini
# karsilastirir.
import os, json, platform, torch

ONCE = {"matmul": torch.backends.cuda.matmul.allow_tf32,
        "cudnn":  torch.backends.cudnn.allow_tf32}
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

DONANIM = {
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "gpu_bellek_gb": round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1)
                     if torch.cuda.is_available() else None,
    "cuda": torch.version.cuda, "torch": torch.__version__,
    "python": platform.python_version(), "matmul_tf32": False, "cudnn_tf32": False,
}
for k, v in DONANIM.items(): print(f"  {k:<16} {v}")

# TF32 varsayilani zaten False ise (PyTorch >= 1.12) bu pin bir sey degistirmez;
# True ise ONCEKI kosular daha dusuk hassasiyette yapilmis demektir.
if ONCE["matmul"]:
    print("\nUYARI: matmul TF32 bu ortamda varsayilan olarak ACIKTI ve simdi")
    print("kapatildi. Onceki kollar TF32 ile egitilmis olabilir; yeni kollarin")
    print("sayilari birkac ondalik kayabilir. R@1 400 ornek uzerinde 0.25 adimli")
    print("oldugu icin genellikle degismez, ama MRR kayabilir.")
else:
    print("\nmatmul TF32 zaten kapaliydi (PyTorch varsayilani) -- onceki kollarla")
    print("ayni hassasiyet. Bu pin yalnizca durumu acik hale getirir.")

YOL = f"{DRIVE_BASE}/ZoomInOut/v5i_results/donanim.json"
if os.path.exists(YOL):
    _e = json.load(open(YOL))
    fark = {k: (_e.get(k), DONANIM[k]) for k in ("gpu", "torch", "cuda", "matmul_tf32")
            if _e.get(k) != DONANIM[k]}
    print(f"\nOnceki kayit: {_e.get('gpu')} | torch {_e.get('torch')}")
    print("  AYNI donanim, sure kolonu homojen." if not fark
          else f"  DEGISMIS: {fark}\n  -> sure kolonu iki donanimi karistirir, dikkat.")
json.dump(DONANIM, open(YOL, "w"), indent=2, ensure_ascii=False)
print(f"Kaydedildi -> v5i_results/donanim.json")
print("\nMakalenin 'Deneysel Kurulum' bolumu bu GPU adini vermelidir.")


In [ ]:
# HÜCRE L4-TIMED: cok-seed dogrulama, KOL BAZINDA SURE OLCUMLU
# ---------------------------------------------------------------------------
# Eski L4 sureyi olcuyor ama yalnizca EKRANA basiyordu ve tum seed'in toplamini
# veriyordu; kol basina ayrisma yoktu, sonuc dosyasina da yazilmiyordu. Bu
# surum her kolu ayri olcup sonuca yaziyor.
# CIKTI YOLU YENI: mevcut multiseed_results.pkl'in UZERINE YAZMAZ.
import os, pickle, time

SEEDS = [42, 123, 777, 2024, 31415]
CKPT_PATH = f"{DRIVE_BASE}/ZoomInOut/v5i_results/multiseed_results_timed.pkl"

# Hangi kollar kosulsun -- makaledeki tablo icin ilk bes yeterli.
# B ve C'yi de istiyorsan True yap (C 75 epoch, yavas).
KOSULACAK = {"A_baseline": True, "A_baseline_random_control": True,
             "B_hardneg": True, "C_instr_oracle": False,
             "D_leakfree": True, "D_SACL_lite_leakfree": True,
             "D_SACL_continuous_leakfree": True}

seed_results = pickle.load(open(CKPT_PATH,"rb")) if os.path.exists(CKPT_PATH) else {}
if seed_results: print(f"Onceki checkpoint: {len(seed_results)} seed tamam.")

def _olc(ad, fn, degerlendir):
    """Bir kolu egit, sureyi olc, metrikleri ve sureyi birlikte dondur."""
    t = time.time()
    m = fn()
    sn = time.time() - t
    r = degerlendir(m)
    r["elapsed_sec"] = sn
    print(f"    {ad:<30} {sn/60:5.1f} dk   R@1={r['R@1']:.1f}%")
    del m
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return r

for seed in SEEDS:
    if seed in seed_results and all(k in seed_results[seed] for k,v in KOSULACAK.items() if v):
        print(f"Seed {seed} zaten tamam, atlaniyor."); continue
    print(f"\n{'='*70}\nSEED {seed}\n{'='*70}")
    t_seed = time.time()
    R = seed_results.get(seed, {})

    if KOSULACAK["A_baseline"] and "A_baseline" not in R:
        R["A_baseline"] = _olc("A_baseline",
            lambda: train_model("plain","plain",epochs=25,val_pairs=val_pairs,seed=seed),
            lambda m: evaluate(m,"plain",test_pairs))

    if KOSULACAK["A_baseline_random_control"] and "A_baseline_random_control" not in R:
        R["A_baseline_random_control"] = _olc("A_baseline_random_control",
            lambda: train_model("plain","plain",epochs=25,val_pairs=val_pairs,
                                 loader=random_loader,seed=seed),
            lambda m: evaluate(m,"plain",test_pairs))
        R["A_baseline_random_control"]["random_pairs_sorted"] = True

    if KOSULACAK["B_hardneg"] and "B_hardneg" not in R:
        R["B_hardneg"] = _olc("B_hardneg",
            lambda: train_model("plain","hn",epochs=25,val_pairs=val_pairs,seed=seed),
            lambda m: evaluate(m,"plain",test_pairs))

    if KOSULACAK["C_instr_oracle"] and "C_instr_oracle" not in R:
        R["C_instr_oracle"] = _olc("C_instr_oracle",
            lambda: train_model("instruction","hn",epochs=75,val_pairs=val_pairs,seed=seed),
            lambda m: evaluate(m,"instruction",test_pairs,forced_mode=None))

    if KOSULACAK["D_leakfree"] and "D_leakfree" not in R:
        R["D_leakfree"] = _olc("D_leakfree",
            lambda: train_model_D_leakfree(epochs=75,seed=seed,val_pairs=val_pairs),
            lambda m: evaluate_self_paired(m,test_pairs))

    if KOSULACAK["D_SACL_lite_leakfree"] and "D_SACL_lite_leakfree" not in R:
        R["D_SACL_lite_leakfree"] = _olc("D_SACL_lite_leakfree",
            lambda: train_d_sacl_lite_leakfree(epochs=75,lambda_seq=0.1,pos_window=2,
                                                seed=seed,val_pairs=val_pairs),
            lambda m: evaluate_self_paired(m,test_pairs))

    if KOSULACAK["D_SACL_continuous_leakfree"] and "D_SACL_continuous_leakfree" not in R:
        R["D_SACL_continuous_leakfree"] = _olc("D_SACL_continuous_leakfree",
            lambda: train_d_sacl_continuous_leakfree(epochs=75,lambda_seq=0.1,tau=5.0,
                                                      seed=seed,val_pairs=val_pairs),
            lambda m: evaluate_self_paired(m,test_pairs))

    seed_results[seed] = R
    pickle.dump(seed_results, open(CKPT_PATH,"wb"))
    print(f"  Seed {seed} tamam ({(time.time()-t_seed)/60:.1f} dk) -> kaydedildi.")

print(f"\n=== TUM SEEDLER TAMAM -> {CKPT_PATH} ===")

# ---------------- ozet tablo: R@1 + sure ----------------
import numpy as np
AD = {"A_baseline_random_control":"Vanilla InfoNCE, rastgele ciftler",
      "A_baseline":"Vanilla InfoNCE, ZoomInOut ciftleri",
      "B_hardneg":"B: hard-negative InfoNCE (mod kosullama yok)",
      "C_instr_oracle":"C: instruction-aware (oracle)",
      "D_leakfree":"COSMOS ZoomInOut",
      "D_SACL_lite_leakfree":"  surum A: basamak cekirdegi w=2",
      "D_SACL_continuous_leakfree":"  surum B: ustel cekirdek tau=5"}
print(f"\n{'kol':<46}{'R@1 (%)':>16}{'MRR':>16}{'sure(dk)':>10}")
print("-"*88)
for k, ad in AD.items():
    v = [seed_results[s][k] for s in SEEDS if s in seed_results and k in seed_results[s]]
    if not v: continue
    r1 = np.array([x["R@1"] for x in v]); mr = np.array([x["MRR"] for x in v])
    dk = np.mean([x["elapsed_sec"] for x in v])/60
    print(f"{ad:<46}{f'{r1.mean():.2f} ± {r1.std(ddof=1):.2f}':>16}"
          f"{f'{mr.mean():.3f} ± {mr.std(ddof=1):.3f}':>16}{dk:>10.1f}")
print("-"*88)
print(f"seed: {SEEDS}   |   test n={len(test_pairs)}")


In [ ]:
# HÜCRE R-FIX: rastgele-çift kontrol kolunu deterministik sonuçlarla tazele
# ---------------------------------------------------------------------------
# Bir onceki oturumda bu kol, generate_random_pairs icindeki list(set) hatasi
# yuzunden tekrar uretilemeyen bir cift kumesiyle egitilmisti (belirti: diger
# dort kol onceki sonuclarla birebir ayni cikti, yalnizca bu kol 50.45 -> 48.60
# kaydi). L3b artik sorted() kullaniyor. Bu hucre, checkpoint'te DUZELTME
# ONCESINDEN kalan seed'leri bulup yalnizca onlari yeniden kosar; duzeltmeden
# sonra egitilmis seed'lere dokunmaz. Kol basina ~0.6 dk.
import time, pickle, numpy as np
from torch.utils.data import DataLoader

random_pairs  = generate_random_pairs(hybrid_index, train_image_ids, len(train_pairs), seed=99)
random_loader = DataLoader(PairDataset(random_pairs), batch_size=BATCH_SIZE,
                           shuffle=True, drop_last=True)

bekleyen = [sd for sd in sorted(seed_results)
            if not seed_results[sd].get("A_baseline_random_control", {}).get("random_pairs_sorted")]
if not bekleyen:
    print("Tum seed'ler zaten deterministik cift kumesiyle kosulmus, atlaniyor.")
else:
    print(f"{len(random_pairs)} rastgele cift (deterministik siralama) -> {len(random_loader)} batch")
    print(f"Yeniden kosulacak seed'ler: {bekleyen}\n")
    for sd in bekleyen:
        t = time.time()
        m = train_model("plain", "plain", epochs=25, val_pairs=val_pairs,
                        loader=random_loader, seed=sd)
        r = evaluate(m, "plain", test_pairs)
        r["elapsed_sec"] = time.time() - t
        r["random_pairs_sorted"] = True
        eski = seed_results[sd].get("A_baseline_random_control", {}).get("R@1")
        seed_results[sd]["A_baseline_random_control"] = r
        print(f"  seed {sd:<6} {r['elapsed_sec']/60:4.1f} dk   R@1={r['R@1']:.1f}%"
              + (f"   (eski: {eski:.1f}%)" if eski is not None else ""))
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        pickle.dump(seed_results, open(CKPT_PATH, "wb"))
    r1 = np.array([seed_results[s]["A_baseline_random_control"]["R@1"] for s in sorted(seed_results)])
    print(f"\nrastgele-cift kontrolu: {r1.mean():.2f} ± {r1.std(ddof=1):.2f}   (kaydedildi)")


In [ ]:
# HÜCRE L5 (DÜZELTİLMİŞ) -- çöken hücrenin yerine bunu koy. EĞİTİM TEKRARI YOK.
# 1) yalnizca GERCEKTEN kosulmus kollar ozetlenir (B/C kapaliydi -> KeyError)
# 2) std artik ddof=1 (orneklem std'si); eski hucre ddof=0 kullanip makaledeki
#    degerlerden daha dar bir std basiyordu (45.75 icin 1.25 yerine 1.40 dogru)
# 3) kol basina ortalama egitim suresi de tabloya ve pkl'e giriyor
import numpy as np, pickle
from scipy import stats

ADAYLAR = ["A_baseline", "A_baseline_random_control", "B_hardneg", "C_instr_oracle",
           "D_leakfree", "D_SACL_lite_leakfree", "D_SACL_continuous_leakfree",
           "D_SACL_continuous_realneg_leakfree"]
SEEDLER  = sorted(seed_results.keys())
variants = [v for v in ADAYLAR if all(v in seed_results[s] for s in SEEDLER)]
eksik    = [v for v in ADAYLAR if v not in variants and any(v in seed_results[s] for s in SEEDLER)]
if eksik: print(f"Kismen kosulmus, atlandi: {eksik}\n")

print(f"{'Varyant':<34} | {'R@1 (ort±std)':<17} | {'MRR (ort±std)':<17} | {'dk':>5}")
print("-" * 86)
MULTISEED_SUMMARY = {}
for v in variants:
    r1  = np.array([seed_results[s][v]["R@1"]  for s in SEEDLER])
    mrr = np.array([seed_results[s][v]["MRR"] for s in SEEDLER])
    sn  = [seed_results[s][v].get("elapsed_sec") for s in SEEDLER]
    dk  = np.mean(sn)/60 if all(x is not None for x in sn) else float("nan")
    MULTISEED_SUMMARY[v] = {"R@1_mean": r1.mean(), "R@1_std": r1.std(ddof=1),
                            "MRR_mean": mrr.mean(), "MRR_std": mrr.std(ddof=1),
                            "train_min_mean": dk, "n_seeds": len(SEEDLER)}
    print(f"{v:<34} | {r1.mean():5.2f}% ± {r1.std(ddof=1):4.2f} | "
          f"{mrr.mean():.3f} ± {mrr.std(ddof=1):.3f} | {dk:5.1f}")
print("-" * 86)
print(f"seed: {SEEDLER}  |  test n={len(test_pairs)}  |  std: ddof=1")

with open(f"{DRIVE_BASE}/ZoomInOut/v5i_results/multiseed_summary_timed.pkl","wb") as f:
    pickle.dump(MULTISEED_SUMMARY, f)
print("Kaydedildi -> v5i_results/multiseed_summary_timed.pkl")

def _t(a, b, etiket):
    if a not in variants or b not in variants:
        print(f"  ({etiket}: kol eksik, atlandi)"); return
    t, p = stats.ttest_rel([seed_results[s][a]["R@1"] for s in SEEDLER],
                           [seed_results[s][b]["R@1"] for s in SEEDLER])
    print(f"  {etiket:<54} t={t:+7.3f}  p={p:.4f}")

print("\nESLESMIS t-TESTLERI (R@1)")
_t("D_leakfree", "A_baseline",                            "COSMOS ZoomInOut vs vanilla (ZoomInOut ciftleri)")
_t("D_leakfree", "A_baseline_random_control",             "COSMOS ZoomInOut vs vanilla (rastgele ciftler)")
_t("A_baseline", "A_baseline_random_control",             "motor-secimi ciftler vs rastgele ciftler")
_t("D_SACL_lite_leakfree", "D_leakfree",                  "surum A (sert pencere) vs zincir kaybi YOK")
_t("D_SACL_continuous_leakfree", "D_leakfree",            "surum B (ustel cekirdek) vs zincir kaybi YOK")
_t("D_SACL_continuous_leakfree", "D_SACL_lite_leakfree",  "surum B vs surum A")


## 20. SACL Ablasyon Tablosu (pos_window × lambda_seq)

In [ ]:
# HÜCRE L6: SACL hiperparametre mini-grid (tek seed, hizli tarama)
ABLATION_GRID = [
    {"pos_window": 1, "lambda_seq": 0.05},
    {"pos_window": 1, "lambda_seq": 0.1},
    {"pos_window": 2, "lambda_seq": 0.05},
    {"pos_window": 2, "lambda_seq": 0.1},
    {"pos_window": 2, "lambda_seq": 0.2},
    {"pos_window": 4, "lambda_seq": 0.1},
]
ABLATION_CKPT = f"{DRIVE_BASE}/ZoomInOut/v5i_results/ablation_results.pkl"
if os.path.exists(ABLATION_CKPT):
    with open(ABLATION_CKPT, "rb") as f:
        ablation_results = pickle.load(f)
else:
    ablation_results = {}

for cfg in ABLATION_GRID:
    key = (cfg["pos_window"], cfg["lambda_seq"])
    if key in ablation_results:
        print(f"{key} zaten tamamlanmış, atlanıyor.")
        continue
    print(f"\n--- pos_window={cfg['pos_window']} lambda_seq={cfg['lambda_seq']} ---")
    m = train_d_sacl_lite_leakfree(epochs=75, lambda_seq=cfg["lambda_seq"], pos_window=cfg["pos_window"], seed=SEED, val_pairs=val_pairs)
    r_val_final = evaluate_self_paired(m, val_pairs)
    r_test = evaluate_self_paired(m, test_pairs)
    ablation_results[key] = {"val": r_val_final, "test": r_test}
    with open(ABLATION_CKPT, "wb") as f:
        pickle.dump(ablation_results, f)
    print(f"  -> val R@1={r_val_final['R@1']:.1f}%  test R@1={r_test['R@1']:.1f}%")

print(f"\n{'pos_window':<12} | {'lambda_seq':<12} | {'val R@1':<9} | {'test R@1':<9} | {'test MRR':<9}")
print("-" * 60)
for (pw, ls), r in sorted(ablation_results.items()):
    print(f"{pw:<12} | {ls:<12} | {r['val']['R@1']:.1f}%    | {r['test']['R@1']:.1f}%     | {r['test']['MRR']:.3f}")
print("\nNOT: en iyi ayari VAL sutununa gore sec (test'e bakarak secim, test-set")
print("tuning sayilir). Sectigin konfigurasyonun TEST sutunu, rapor edilecek")
print("nihai sayidir.")


## 20b. D+SACL-continuous Tau Taraması

D+SACL-continuous, tek koşuda D+SACL-lite'ı anlamlı farkla geçmişti
(t=3.27, p=0.03, tau=3.0 ile). tau, "kavramsal mesafe"nin ne kadar hızlı
sönümlendiğini kontrol ediyor -- küçük tau = sadece çok yakın adımlar
önemli (sert pencereye yaklaşır), büyük tau = zincirin tamamı neredeyse
eşit ağırlıklı. En iyi tau'yu bulmak için tarama yapıyoruz.

In [ ]:
# HÜCRE L6c: D+SACL-continuous tau taramasi
TAU_GRID = [1.0, 2.0, 3.0, 5.0, 8.0]
TAU_ABLATION_CKPT = f"{DRIVE_BASE}/ZoomInOut/v5i_results/tau_ablation_results.pkl"
if os.path.exists(TAU_ABLATION_CKPT):
    with open(TAU_ABLATION_CKPT, "rb") as f:
        tau_ablation_results = pickle.load(f)
else:
    tau_ablation_results = {}

for tau in TAU_GRID:
    if tau in tau_ablation_results:
        print(f"tau={tau} zaten tamamlanmış, atlanıyor.")
        continue
    print(f"\n--- tau={tau} ---")
    m = train_d_sacl_continuous_leakfree(epochs=75, lambda_seq=0.1, tau=tau, seed=SEED, val_pairs=val_pairs)
    r_val_final = evaluate_self_paired(m, val_pairs)
    r_test = evaluate_self_paired(m, test_pairs)
    tau_ablation_results[tau] = {"val": r_val_final, "test": r_test}
    with open(TAU_ABLATION_CKPT, "wb") as f:
        pickle.dump(tau_ablation_results, f)
    r1v = r_val_final["R@1"]
    r1t = r_test["R@1"]
    print(f"  -> val R@1={r1v:.1f}%  test R@1={r1t:.1f}%")

print(f"{'tau':<8} | {'val R@1':<9} | {'test R@1':<9} | {'test MRR':<9}")
print("-" * 45)
for tau, r in sorted(tau_ablation_results.items()):
    rv1 = r["val"]["R@1"]
    rt1 = r["test"]["R@1"]
    rtm = r["test"]["MRR"]
    print(f"{tau:<8} | {rv1:.1f}%    | {rt1:.1f}%     | {rtm:.3f}")
print()
print("NOT: en iyi tau degerini VAL sutununa gore sec (test-set tuning olmasin).")


## 21. Nitel Retrieval Örnekleri (paper için görsel örnekler)

In [ ]:
# HÜCRE L7: en iyi modelle (D+SACL-lite leak-free) birkac ornek retrieval goster
import random

QUAL_PATH = f"{DRIVE_BASE}/ZoomInOut/v5i_results/qualitative_examples.pkl"

if os.path.exists(QUAL_PATH):
    print("=== Nitel örnekler zaten üretilmişti, ATLANIYOR ===")
    with open(QUAL_PATH, "rb") as f:
        QUALITATIVE_EXAMPLES = pickle.load(f)
    for ex in QUALITATIVE_EXAMPLES:
        rank = ex["true_rank"]
        status = "✅ DOĞRU (R@1)" if rank == 1 else (f"R@{rank}" if rank <= 10 else f"BAŞARISIZ (rank={rank})")
        print(f"test[{ex['test_idx']}]: gerçek eşleşme rank={rank}  {status}")
elif "_m_dsacl_test" not in globals():
    print("⚠️  _m_dsacl_test bellekte yok (Bölüm 18 atlandı) VE önceden kaydedilmiş")
    print("    nitel örnek de yok -- bu bölüm atlanıyor, sorun değil (paper için")
    print("    zorunlu değil, sadece görsel örnek amaçlı).")
else:
    best_model = _m_dsacl_test  # Bölüm 18'de eğitilen tek-koşu D+SACL-lite modeli
    best_model.eval()

    img_all = torch.tensor(np.stack([np.asarray(p["img_embedding"], dtype=np.float32) for p in test_pairs])).to(device)
    txt_all = torch.tensor(np.stack([np.asarray(p["txt_embedding"], dtype=np.float32) for p in test_pairs])).to(device)
    with torch.no_grad():
        mode_vec_img, _ = best_model.infer_mode(img_all, img_all)
        mode_vec_txt, _ = best_model.infer_mode(txt_all, txt_all)
        z_img = best_model.img_proj(img_all, mode_vec_img)
        z_txt = best_model.txt_proj(txt_all, mode_vec_txt)
    sim = (z_txt @ z_img.T).cpu().numpy()

    QUALITATIVE_EXAMPLES = []
    sample_idxs = random.sample(range(len(test_pairs)), min(10, len(test_pairs)))
    for i in sample_idxs:
        order = np.argsort(-sim[i])
        rank = int(np.where(order == i)[0][0]) + 1
        top5 = order[:5].tolist()
        QUALITATIVE_EXAMPLES.append({
            "test_idx": i, "true_rank": rank, "top5_idx": top5,
            "coco_id": test_pairs[i].get("coco_id"),
            "cross_modal_sim": test_pairs[i].get("cross_modal_sim"),
        })
        status = "✅ DOĞRU (R@1)" if rank == 1 else (f"R@{rank}" if rank <= 10 else f"BAŞARISIZ (rank={rank})")
        print(f"test[{i}]: gerçek eşleşme rank={rank}  {status}")

    with open(QUAL_PATH, "wb") as f:
        pickle.dump(QUALITATIVE_EXAMPLES, f)
    print("\nKaydedildi -> v5i_results/qualitative_examples.pkl")
    print("\nNot: gerçek görselleri/metinleri raporda göstermek için test_pairs[i] içindeki")
    print("kaynak image_id/coco_id ile orijinal COCO görsellerini eşleştirip indirmen gerekir.")


## 22. ARASTIRMA: Motor hala neden kismen rastgeleden zayif?

Hub-etkisi duzeltmesi farki 16.3 puandan 4.7 puana indirdi ama kapatmadi.
Hipotez: motor, `img_shift` adimlarinda KASITLI olarak "farkli, dusuk-IoU"
bir nesne secip (hard-negative uretmek icin), bu kirpimi bir SONRAKI pozitif
eslesmenin CAPASI yapiyor olabilir -- COCO caption'in tanimladigi ANA
nesneden farkli/cevresel bir kirpim, dogal olarak daha zayif eslesir.

In [ ]:
# ARASTIRMA HUCRESI: motor-secimli ciftler neden hala rastgeleden zayif?
import numpy as np

def pair_img_txt_sim(pairs):
    img = np.stack([p["img_embedding"] for p in pairs])
    txt = np.stack([p["txt_embedding"] for p in pairs])
    img = img / (np.linalg.norm(img, axis=1, keepdims=True) + 1e-9)
    txt = txt / (np.linalg.norm(txt, axis=1, keepdims=True) + 1e-9)
    return np.sum(img * txt, axis=1)

motor_sims = pair_img_txt_sim(train_pairs)
random_sims = pair_img_txt_sim(random_pairs)

print("=== GORSEL-METIN ESLESME KALITESI: MOTOR vs RASTGELE ===")
print(f"Motor-secimli  cift: ort. cosine sim = {motor_sims.mean():.4f} (std={motor_sims.std():.4f}, n={len(motor_sims)})")
print(f"Rastgele-secim cift: ort. cosine sim = {random_sims.mean():.4f} (std={random_sims.std():.4f}, n={len(random_sims)})")

from scipy import stats as _stats
tstat, pval = _stats.ttest_ind(motor_sims, random_sims, equal_var=False)
print(f"\nBagimsiz iki-orneklem t-test: t={tstat:.3f}, p={pval:.4g}")
if pval < 0.05 and motor_sims.mean() < random_sims.mean():
    print("-> Motor-secimli ciftlerin gorsel-metin uyumu ISTATISTIKSEL OLARAK DAHA ZAYIF.")
    print("   Bu, img_shift\'in kasitli olarak farkli/cevresel nesneler secmesinin,")
    print("   kalan performans farkina katkida bulunabilecegi hipotezini destekler.")
elif pval < 0.05:
    print("-> Motor-secimli ciftler ISTATISTIKSEL OLARAK DAHA IYI uyumlu -- hipotezi CURUTUR.")
else:
    print("-> Anlamli fark yok -- gorsel-metin uyum kalitesi motor/rastgele arasinda farksiz,")
    print("   kalan fark baska bir kaynaktan geliyor olmali (orn. kalan hub yogunlasmasi).")

# EK: train_sequences bellekte kaldiysa, adim-turune gore de kirilim yap
if "train_sequences" in globals():
    step_type_sims = {"start": [], "txt_to_img": [], "img_shift": []}
    for chain in train_sequences:
        for i, step in enumerate(chain):
            if step.modality == "image" and step.action in step_type_sims:
                if i + 1 < len(chain) and chain[i + 1].action == "img_to_txt":
                    txt_step = chain[i + 1]
                    if txt_step.similarity is not None:
                        step_type_sims[step.action].append(txt_step.similarity)
    print("\n=== Adim turune gore HAM benzerlik (COCO-caption\'dan ONCEKI, egilim gostergesi) ===")
    for k, v in step_type_sims.items():
        if v:
            print(f"  {k:<12}: ort={np.mean(v):.4f}  n={len(v)}")
else:
    print("\n(train_sequences bellekte yok -- adim-turu kirilimi atlandi, sadece yukaridaki")
    print(" genel motor-vs-rastgele karsilastirmasi gecerli.)")


## 24. Hoca Talebi: SADECE D+SACL, 5-Seed, Epoch/Süre Takipli

Hocanın isteği: (1) sadece D+SACL yöntemine odaklan (diğer varyantları
tekrar koşturma), (2) 5 seed ortalaması al, (3) her seed'in kaç epoch
çalıştığını (veriyi kaç kez gördüğünü) kaydet, (4) süreleri ekle, (5)
"veriyi çok görmek işe yaradı mı" sorusuna (epoch sayısı ile R@1 arasında
bir ilişki var mı) cevap ver.

Bu hücreler `train_d_sacl_lite_leakfree`'nin AYNI eğitim mantığının bir
kopyasıdır -- sadece epoch/süre izleme eklenmiş hali. Mevcut çok-seed
(HÜCRE L4/L5) hücrelerine DOKUNULMADI, onlar hâlâ çalışır durumda.

In [ ]:
# HÜCRE D_SACL_FOCUS1: D+SACL-lite egitim fonksiyonu (epoch/sure izlemeli)
import time

def train_d_sacl_lite_leakfree_timed(epochs=75, lambda_seq=0.1, pos_window=2, seed=None,
                                      val_pairs=None, patience=4, eval_every=5):
    """train_d_sacl_lite_leakfree ile AYNI egitim mantigi -- ek olarak kac
    epoch calistigini, en iyi val sonucunun hangi epoch'ta bulundugunu, ve
    toplam egitim suresini (saniye) de dondurur."""
    t0 = time.time()
    torch.manual_seed(seed if seed is not None else SEED)
    model = SelfInferringHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    chain_iter = itertools.cycle(chain_loader)
    best_state = copy.deepcopy(model.state_dict())
    best_val_r1 = -1.0
    best_epoch = 0
    checks_since_improve = 0
    epochs_run = 0

    for epoch in range(epochs):
        epochs_run = epoch + 1
        model.train()
        epoch_loss = epoch_aux = epoch_seq = 0.0
        for batch in train_loader:
            img = batch["img"].to(device); txt = batch["txt"].to(device)
            hn = batch["hn"].to(device); has_hn = batch["has_hn"].to(device)
            mode_idx = batch["mode_idx"].to(device)

            optimizer.zero_grad()
            mode_vec_img, mode_logits = model.infer_mode(img, img)
            mode_vec_txt, _ = model.infer_mode(txt, txt)
            z_img = model.img_proj(img, mode_vec_img)
            z_txt = model.txt_proj(txt, mode_vec_txt)
            z_hn = model.project_image_with_vec(hn, mode_vec_img) if has_hn.sum() > 0 else None

            base_loss = hn_info_nce(z_img, z_txt, z_hn=z_hn, has_hn=has_hn)
            mask = has_hn.bool()
            aux = F.cross_entropy(mode_logits[mask], mode_idx[mask]) if mask.sum() > 0 else torch.tensor(0.0, device=device)

            cb = next(chain_iter).to(device)
            z_chain = project_chain_batch_self_infer(model, cb)
            l_seq, _ = sequence_aware_loss(z_chain, cb, pos_window=pos_window,
                                            lambda_multipos=LAMBDA_MULTIPOS, lambda_rank=LAMBDA_RANK,
                                            margin=RANK_MARGIN, same_modality_only=True)

            loss = base_loss + MODE_AUX_WEIGHT * aux + lambda_seq * l_seq
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); epoch_aux += aux.item(); epoch_seq += l_seq.item()
        scheduler.step()
        if (epoch + 1) % 25 == 0 or epoch == 0:
            n = len(train_loader)
            print(f"  [D+SACL-lite-timed] epoch {epoch+1}/{epochs} loss={epoch_loss/n:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

        if val_pairs is not None and ((epoch + 1) % eval_every == 0 or epoch + 1 == epochs):
            r_val = evaluate_self_paired(model, val_pairs)
            if r_val["R@1"] > best_val_r1:
                best_val_r1 = r_val["R@1"]
                best_epoch = epoch + 1
                best_state = copy.deepcopy(model.state_dict())
                checks_since_improve = 0
            else:
                checks_since_improve += 1
            if checks_since_improve >= patience:
                print(f"  [erken durdurma] epoch {epoch+1}'de durduruldu (en iyi val R@1=%{best_val_r1:.1f}, epoch {best_epoch}).")
                break

    elapsed = time.time() - t0
    if val_pairs is not None:
        model.load_state_dict(best_state)
    return model, {"epochs_run": epochs_run, "best_epoch": best_epoch,
                    "best_val_r1": best_val_r1, "elapsed_sec": elapsed}


In [ ]:
# HÜCRE D_SACL_FOCUS2: 5-seed kosusu (checkpoint'li, kesintiye dayanikli)
DSACL_FOCUS_SEEDS = [42, 123, 777, 2024, 31415]
DSACL_FOCUS_CKPT = f"{DRIVE_BASE}/ZoomInOut/v5i_results/dsacl_focus_results.pkl"

os.makedirs(f"{DRIVE_BASE}/ZoomInOut/v5i_results", exist_ok=True)
if os.path.exists(DSACL_FOCUS_CKPT):
    with open(DSACL_FOCUS_CKPT, "rb") as f:
        dsacl_focus_results = pickle.load(f)
    print(f"Onbellek bulundu: {len(dsacl_focus_results)} seed zaten tamamlanmis.")
else:
    dsacl_focus_results = {}

for sd in DSACL_FOCUS_SEEDS:
    if sd in dsacl_focus_results:
        print(f"seed={sd} zaten var, atlaniyor.")
        continue
    print(f"\n=== D+SACL-lite, seed={sd} ===")
    model, meta = train_d_sacl_lite_leakfree_timed(seed=sd, val_pairs=val_pairs)
    r_test = evaluate_self_paired(model, test_pairs)
    meta["R@1"] = r_test["R@1"]; meta["R@5"] = r_test["R@5"]
    meta["R@10"] = r_test["R@10"]; meta["MRR"] = r_test["MRR"]
    dsacl_focus_results[sd] = meta
    print(f"  epochs_calisan={meta['epochs_run']} en_iyi_epoch={meta['best_epoch']} "
          f"sure={meta['elapsed_sec']:.1f}sn  R@1={meta['R@1']:.1f}%  MRR={meta['MRR']:.3f}")
    with open(DSACL_FOCUS_CKPT, "wb") as f:
        pickle.dump(dsacl_focus_results, f)

print(f"\nTum seedler tamam ({len(dsacl_focus_results)}/{len(DSACL_FOCUS_SEEDS)}).")


In [ ]:
# HÜCRE D_SACL_FOCUS3: ozet tablo + "veriyi cok gormek ise yaradi mi" analizi
import numpy as np
from scipy import stats as _stats

seeds_sorted = sorted(dsacl_focus_results.keys())
r1s = np.array([dsacl_focus_results[s]["R@1"] for s in seeds_sorted])
mrrs = np.array([dsacl_focus_results[s]["MRR"] for s in seeds_sorted])
epochs_run = np.array([dsacl_focus_results[s]["epochs_run"] for s in seeds_sorted])
best_epochs = np.array([dsacl_focus_results[s]["best_epoch"] for s in seeds_sorted])
times = np.array([dsacl_focus_results[s]["elapsed_sec"] for s in seeds_sorted])

print("=== D+SACL-lite: 5-Seed Ozet Tablosu (epoch/sure ile) ===\n")
print(f"{'Seed':<8} | {'Calisan epoch':<14} | {'En iyi epoch':<13} | {'Sure (sn)':<10} | {'R@1':<7} | {'MRR':<6}")
print("-" * 75)
for s in seeds_sorted:
    m = dsacl_focus_results[s]
    print(f"{s:<8} | {m['epochs_run']:<14} | {m['best_epoch']:<13} | {m['elapsed_sec']:<10.1f} | {m['R@1']:<6.1f}% | {m['MRR']:.3f}")
print("-" * 75)
print(f"{'ORTALAMA':<8} | {epochs_run.mean():<14.1f} | {best_epochs.mean():<13.1f} | {times.mean():<10.1f} | {r1s.mean():<6.1f}% | {mrrs.mean():.3f}")
print(f"{'STD':<8} | {epochs_run.std():<14.1f} | {best_epochs.std():<13.1f} | {times.std():<10.1f} | {r1s.std():<6.1f}% | {mrrs.std():.3f}")
print(f"\nToplam sure (5 seed birlikte): {times.sum()/60:.1f} dakika (ort. {times.mean()/60:.1f} dk/seed)")

print("\n=== 'Veriyi cok gormek ise yaradi mi?' -- en_iyi_epoch vs R@1 korelasyonu ===")
if len(seeds_sorted) >= 3:
    corr, pval = _stats.pearsonr(best_epochs, r1s)
    print(f"Pearson r={corr:.3f}, p={pval:.4f}  (n={len(seeds_sorted)} -- KUCUK ornek, guc dusuk, temkinli yorumla)")
    if pval < 0.05 and corr > 0:
        print("-> Anlamli POZITIF iliski: daha GEC en-iyi-checkpoint bulan (daha cok epoch gormus)")
        print("   seed'ler daha yuksek R@1 aliyor -- 'veriyi daha cok gormek' isaretleri var.")
    elif pval < 0.05 and corr < 0:
        print("-> Anlamli NEGATIF iliski: erken duran seed'ler daha iyi -- daha COK epoch")
        print("   FAYDALI degil, asiri ogrenme (overfit) riski isaretleri olabilir.")
    else:
        print("-> Anlamli bir iliski YOK -- n=5 ile epoch sayisi/basari arasinda net bir")
        print("   sonuc cikaramiyoruz (bu, dogru ve durust bir sonuc -- daha fazla seed gerekir).")
else:
    print("En az 3 seed gerekli, henuz yetersiz.")

with open(f"{DRIVE_BASE}/ZoomInOut/v5i_results/dsacl_focus_summary.pkl", "wb") as f:
    pickle.dump({
        "per_seed": dsacl_focus_results,
        "r1_mean": float(r1s.mean()), "r1_std": float(r1s.std()),
        "mrr_mean": float(mrrs.mean()), "mrr_std": float(mrrs.std()),
        "epochs_run_mean": float(epochs_run.mean()), "best_epoch_mean": float(best_epochs.mean()),
        "total_time_sec": float(times.sum()), "avg_time_sec": float(times.mean()),
    }, f)
print("\nKaydedildi -> v5i_results/dsacl_focus_summary.pkl")


## 25. PENCERE GENİŞLİĞİ DENEYİ: SACL hard window, w = 2 vs w = 4

Makalede raporlanan D+SACL hard window sonucu `pos_window=2` ile alınmıştır;
altyapıdaki `POS_WINDOW = 4` sabiti eğitimde kullanılmamaktadır. Bu bölüm iki
değeri **5 seed** ile doğrudan karşılaştırır, böylece pencere genişliğinin
etkisi tek seed'lik bir taramaya değil istatistiksel bir teste dayanır.

Çıktılar `v5i_results/window_ablation_results.pkl` dosyasına yazılır; mevcut
sonuç dosyalarına dokunulmaz.


In [ ]:
# HÜCRE WIN1: SACL hard window genisligi w=2 vs w=4, 5 seed
# Ayni veri, ayni mimari, ayni seed'ler; tek degisken pos_window.
for _n in ["train_d_sacl_lite_leakfree", "evaluate_self_paired", "val_pairs", "test_pairs",
           "train_loader", "chain_loader"]:
    assert _n in globals(), f"ONKOSUL EKSIK: {_n}. Notebook'u sirayla calistir."

WINDOW_GRID   = [2, 4]
WINDOW_SEEDS  = [42, 123, 777, 2024, 31415]
WINDOW_CKPT   = f"{DRIVE_BASE}/ZoomInOut/v5i_results/window_ablation_results.pkl"
os.makedirs(f"{DRIVE_BASE}/ZoomInOut/v5i_results", exist_ok=True)

if os.path.exists(WINDOW_CKPT):
    with open(WINDOW_CKPT, "rb") as f:
        window_results = pickle.load(f)
    print(f"Onbellek bulundu: {sum(len(v) for v in window_results.values())} kosu tamam.")
else:
    window_results = {w: {} for w in WINDOW_GRID}

for w in WINDOW_GRID:
    window_results.setdefault(w, {})
    for sd in WINDOW_SEEDS:
        if sd in window_results[w]:
            print(f"w={w} seed={sd} zaten var, atlaniyor."); continue
        print(f"\n=== SACL hard window, w={w}, seed={sd} ===")
        model = train_d_sacl_lite_leakfree(epochs=75, lambda_seq=0.1, pos_window=w,
                                            seed=sd, val_pairs=val_pairs)
        r = evaluate_self_paired(model, test_pairs)
        window_results[w][sd] = {"R@1": r["R@1"], "R@5": r["R@5"],
                                  "R@10": r["R@10"], "MRR": r["MRR"]}
        print(f"  R@1={r['R@1']:.1f}%  R@5={r['R@5']:.1f}%  MRR={r['MRR']:.3f}")
        with open(WINDOW_CKPT, "wb") as f:
            pickle.dump(window_results, f)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print(f"\nTamam -> {WINDOW_CKPT}")


In [ ]:
# HÜCRE WIN2: w=2 vs w=4 ozet + istatistik
from scipy import stats as _st
with open(WINDOW_CKPT, "rb") as f:
    window_results = pickle.load(f)

_seeds = sorted(set(window_results[2]) & set(window_results[4]))
print(f"Ortak seed: {_seeds}  (n={len(_seeds)})\n")
print(f"{'w':<6} {'R@1 (%)':>18} {'R@5 (%)':>18} {'MRR':>18}")
print("-" * 64)

_r1 = {}
for w in [2, 4]:
    r1  = np.array([window_results[w][s]["R@1"]  for s in _seeds], dtype=float)
    r5  = np.array([window_results[w][s]["R@5"]  for s in _seeds], dtype=float)
    mrr = np.array([window_results[w][s]["MRR"]  for s in _seeds], dtype=float)
    _r1[w] = r1
    c1 = f"{r1.mean():.2f} ± {r1.std(ddof=1):.2f}"
    c2 = f"{r5.mean():.2f} ± {r5.std(ddof=1):.2f}"
    c3 = f"{mrr.mean():.3f} ± {mrr.std(ddof=1):.3f}"
    print(f"{w:<6} {c1:>18} {c2:>18} {c3:>18}")

t, p = _st.ttest_rel(_r1[4], _r1[2])
print(f"\nFark (w=4 eksi w=2): {_r1[4].mean() - _r1[2].mean():+.2f} puan")
print(f"Eslestirilmis t-test: t={t:.3f}  p={p:.4f}  ->  "
      f"{'ANLAMLI' if p < 0.05 else 'anlamli fark yok'}")
print("\nMakalede raporlanan kosu w=2 ile yapilmistir. Fark anlamsizsa bu secim")
print("sonuclari etkilemiyor demektir ve makalede oyle yazilabilir.")

with open(f"{DRIVE_BASE}/ZoomInOut/v5i_results/window_ablation_summary.pkl", "wb") as f:
    pickle.dump({"per_seed": window_results, "seeds": _seeds,
                 "paired_t": float(t), "paired_p": float(p)}, f)
print("✅ Ozet kaydedildi.")


---
### ÇALIŞTIRMA SIRASI (bu oturum için)

**Run All YAPMA.** 58-71 arası hücreler önceki taramalar; saatler sürer ve
sonuçları zaten elimizde.

| Hücre | Ne yapar | Süre |
|---|---|---|
| 52 | `generate_random_pairs` determinizm düzeltmesi | anında |
| 54 | GPU-KAYIT, donanımı kaydeder ve TF32'yi sabitler | saniyeler |
| 55 | L4-TIMED, eksik kalan `B_hardneg` kolunu ekler | ~3 dk |
| 56 | R-FIX, rastgele-çift kolunu düzeltilmiş çiftlerle tazeler | ~3 dk |
| 57 | L5, özet tablo + eşleşmiş t-testleri | anında |
| 74 | SACL-1, çatışmayı ölçer, eğitim yok | saniyeler |
| 75 | SACL-2, çatışma deneyi, 4 kol x 5 seed | ~2.5 sa |

**Önbellek şart.** Hücre 7 "Train zaten var, diskten yükleniyor" yazmalı. Sıfırdan
üretirse veri kümesi değişir, çünkü `generate_sequences` de `list(set)` kullanıyor;
o durumda makaledeki bütün sayılar geçersiz olur.

Hücre 75'te kollar, asıl soruyu yanıtlayanlar önce gelecek şekilde sıralandı:
ilk ~80 dakikada `hn_DISLANMIS` kollarının ikisi de biter. Runtime kopsa bile
her seed sonrası Drive'a yazıldığı için kaldığı yerden devam eder.

---
## 26. SACL ÇATIŞMA DENEYİ

Zincirde `img_shift` ile üretilen zorlu negatif `chain[i+1]`, çapa görselle
`chain[i-1]` **aynı ana görselden** gelir ve ondan **iki adım** uzaktadır:

```python
if i + 1 < len(chain) and chain[i + 1].action == "img_shift":
    if (shift_step.image_id == img_step.image_id
            and getattr(shift_step, "verified_label", "positive") == "hard_negative"):
```

`pos_window=2` ile bu çift, zincir kaybının pozitif kümesine tam olarak girer.
Zincir kaybı "bu ikisini yaklaştır" derken HN-InfoNCE aynı düğümü ilgili
metinden uzaklaştırır; iki terim ters yöne çekiyor olabilir.

Ayrıca `build_chain_records` yalnızca `step`, `modality` ve `embedding`
saklayıp `verified_label` alanını düşürdüğü için, zincir kaybı bir düğümün
zorlu negatif olarak işaretlendiğini hiç göremiyor. Aşağıdaki hücreler bu
etiketi geri koyup zorlu negatifleri pozitif kümesinden çıkarır.

In [ ]:
# HÜCRE SACL-1 (TEŞHİS, ~saniyeler, eğitim YOK)
# ---------------------------------------------------------------------------
# ONEMLI: onbellek yolunda (TRAIN_PAIRS_PATH + CHAIN_RECORDS_PATH mevcut)
# `train_sequences` HIC OLUSMAZ -- hucre 7 bunu acikca soyluyor. Dolayisiyla
# zorlu negatif etiketi verified_label'dan okunamaz. Etiketi YENIDEN URETMEDEN
# kurtarmak icin train_pairs onbellegindeki hard_negative_embedding dizileri
# baytlarina gore eslestirilir: bir zincir dugumunun gommesi bu kumedeyse o
# dugum zorlu negatif olarak isaretlenmis demektir.
#
# Bu yol ONEMLI cunku sekanslari yeniden uretmek veri kumesini DEGISTIRIR
# (generate_sequences da `image_ids = list(image_ids)` kullaniyor ve
# train_image_ids bir SET), yani makaledeki tum sayilar gecersiz olur.
import numpy as np, torch

def _bayt(e):
    return np.ascontiguousarray(np.asarray(e, dtype=np.float32)).tobytes()

if "train_sequences" in globals():
    print("train_sequences bellekte -> etiket dogrudan verified_label'dan okunuyor.")
    _img = {p.patch_id: p.embedding for p in hybrid_index.image_patches}
    _txt = {p.patch_id: p.embedding for p in hybrid_index.text_patches}
    CHAIN_HN = []
    for chain in train_sequences:
        nodes = []
        for st in chain:
            e = _img.get(st.patch_id) if st.modality == "image" else _txt.get(st.patch_id)
            if e is None: continue
            nodes.append({"step": st.step,
                          "modality": 0 if st.modality == "image" else 1,
                          "is_hn": 1 if getattr(st, "verified_label", "positive") == "hard_negative" else 0,
                          "embedding": np.asarray(e, dtype=np.float32)})
        if len(nodes) >= 8: CHAIN_HN.append(nodes)
else:
    print("train_sequences YOK (onbellek yolu) -> etiket train_pairs'ten bayt "
          "eslestirmeyle kurtariliyor.")
    HN_BAYT = {_bayt(p["hard_negative_embedding"]) for p in train_pairs
               if p.get("hard_negative_embedding") is not None}
    print(f"  train_pairs icinde {len(HN_BAYT)} FARKLI zorlu negatif yama gommesi var "
          f"({sum(1 for p in train_pairs if p.get('hard_negative_embedding') is not None)} cift)")
    CHAIN_HN = [[{"step": n["step"], "modality": n["modality"],
                  "embedding": np.asarray(n["embedding"], dtype=np.float32),
                  "is_hn": int(_bayt(n["embedding"]) in HN_BAYT)} for n in c]
                for c in train_chain_records]

n_dugum = sum(len(c) for c in CHAIN_HN)
n_hn    = sum(n["is_hn"] for c in CHAIN_HN for n in c)
print(f"\n{len(CHAIN_HN)} zincir | {n_dugum} dugum | zorlu negatif etiketli {n_hn} "
      f"(%{n_hn/n_dugum*100:.1f})")
assert n_hn > 0, ("Hic zorlu negatif etiketli dugum bulunamadi. Bayt eslestirmesi "
                  "tutmadi -- SACL-2'yi calistirma, once bunu konus.")

# UYARI: bayt eslestirme, ayni yamanin baska bir zincirde pozitif olarak
# gectigi durumlarda da isaretler. Bu bir ASIRI-isaretlemedir, ama o yama
# gercekten bir zorlu negatif oldugu icin dislanmasi savunulabilir.

def agirlik_payi(cekirdek, ad):
    top = hn_pay = 0.0
    for c in CHAIN_HN:
        s = torch.tensor([x["step"] for x in c]); m = torch.tensor([x["modality"] for x in c])
        h = torch.tensor([x["is_hn"] for x in c], dtype=torch.bool)
        d  = (s.unsqueeze(0) - s.unsqueeze(1)).abs()
        oz = torch.eye(len(c), dtype=torch.bool)
        uy = (m.unsqueeze(0) == m.unsqueeze(1)) & (~oz)      # same_modality_only=True
        w  = cekirdek(d, uy)
        top    += w.sum().item()
        hn_pay += (w * (h.unsqueeze(0) | h.unsqueeze(1)).float()).sum().item()
    pay = hn_pay / top
    print(f"{ad:<36} pozitif agirligin %{pay*100:5.1f}'i zorlu negatif dugumlere dusuyor")
    return pay

print("\nZINCIR KAYBI POZITIF AGIRLIK DAGILIMI")
print("-" * 82)
PAY_A = agirlik_payi(lambda d, u: ((d > 0) & (d <= 2) & u).float(),        "surum A (sert pencere w=2)")
PAY_B = agirlik_payi(lambda d, u: torch.exp(-d.float() / 5.0) * u.float(), "surum B (ustel cekirdek tau=5)")
print("-" * 82)
print("Bu pay, zincir kaybinin HN-InfoNCE terimiyle TERS yone cektigi kismin")
print("buyuklugudur. Sifira yakin degilse iki terim birbirini kismen iptal eder.")


In [ ]:
# HÜCRE SACL-2 (ÇATIŞMA DENEYİ) -- SACL-1'den SONRA çalıştırılır
# Zorlu negatif etiketli düğümler zincir kaybının POZİTİF KÜMESİNDEN çıkarılır.
#   Katki cikarsa -> catisma hipotezi dogrulanir, SACL kurtarilir.
#   Cikmazsa      -> SACL'in katki vermemesi cekirdek/pencere meselesi degildir.
# Kollar, asil soruyu yanitlayanlar ONCE gelecek sekilde siralandi.
import os, pickle, time, copy, itertools
import numpy as np, torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

CKPT_SACL = f"{DRIVE_BASE}/ZoomInOut/v5i_results/sacl_catisma_results.pkl"
SEEDS_SACL = [42, 123, 777, 2024, 31415]

class ChainBatchHN:
    def __init__(s, emb, modality, mask, step_idx, is_hn):
        s.emb, s.modality, s.mask, s.step_idx, s.is_hn = emb, modality, mask, step_idx, is_hn
    def to(s, d):
        return ChainBatchHN(s.emb.to(d), s.modality.to(d), s.mask.to(d),
                            s.step_idx.to(d), s.is_hn.to(d))

class ChainDatasetHN(Dataset):
    def __init__(s, recs, max_len=SACL_MAX_LEN): s.c, s.L = recs, max_len
    def __len__(s): return len(s.c)
    def __getitem__(s, i):
        n   = s.c[i][:s.L]
        emb = torch.zeros(s.L, EMBED_DIM);        mod = torch.zeros(s.L, dtype=torch.long)
        msk = torch.zeros(s.L, dtype=torch.bool); stp = torch.zeros(s.L, dtype=torch.long)
        hn  = torch.zeros(s.L, dtype=torch.bool)
        for k, x in enumerate(n):
            emb[k] = torch.tensor(x["embedding"]); mod[k] = x["modality"]
            stp[k] = x["step"]; msk[k] = True;     hn[k]  = bool(x["is_hn"])
        return emb, mod, msk, stp, hn

chain_loader_hn = DataLoader(
    ChainDatasetHN(CHAIN_HN), batch_size=CHAIN_BATCH_SIZE, shuffle=True, drop_last=True,
    collate_fn=lambda b: ChainBatchHN(*[torch.stack([x[i] for x in b]) for i in range(5)]))
print(f"chain_loader_hn: {len(chain_loader_hn)} batch")

def sacl_maskeli(z, batch, cekirdek, temp=TEMP, lambda_multipos=LAMBDA_MULTIPOS,
                 lambda_rank=LAMBDA_RANK, margin=RANK_MARGIN, hn_disla=True):
    """sequence_aware_loss ile ayni yapi; tek eklenti hn_disla bayragi."""
    B, L, _ = z.shape; dev = z.device
    fz, fc, fs, fm, fh = [], [], [], [], []
    for b in range(B):
        v = batch.mask[b]; n = int(v.sum().item())
        if n == 0: continue
        fz.append(z[b, v]); fc.append(torch.full((n,), b, device=dev, dtype=torch.long))
        fs.append(batch.step_idx[b, v]); fm.append(batch.modality[b, v]); fh.append(batch.is_hn[b, v])
    fz = torch.cat(fz); fc = torch.cat(fc); fs = torch.cat(fs)
    fm = torch.cat(fm); fh = torch.cat(fh); N = fz.size(0)

    sim = fz @ fz.T / temp
    d   = (fs.unsqueeze(0) - fs.unsqueeze(1)).abs()
    oz  = torch.eye(N, dtype=torch.bool, device=dev)
    uy  = (fc.unsqueeze(0) == fc.unsqueeze(1)) & (fm.unsqueeze(0) == fm.unsqueeze(1)) & (~oz)
    w   = cekirdek(d, uy)
    if hn_disla:
        w = w * (~(fh.unsqueeze(0) | fh.unsqueeze(1))).float()

    lg  = sim.masked_fill(oz, float("-inf"))
    lp  = (lg - torch.logsumexp(lg, dim=1, keepdim=True)).masked_fill(oz, 0.0)
    per = -(lp * w).sum(dim=1) / w.sum(dim=1).clamp(min=1e-8)
    var = w.sum(dim=1) > 1e-6
    l_mp = per[var].mean() if var.any() else torch.zeros((), device=dev)

    rt = []
    for b in range(B):
        ib = (fc == b).nonzero(as_tuple=True)[0]
        if hn_disla: ib = ib[~fh[ib]]
        if ib.numel() < 3: continue
        for a in ib[torch.randperm(ib.numel(), device=dev)[:6]]:
            ot = ib[(ib != a) & (fm[ib] == fm[a])]
            if ot.numel() < 2: continue
            dd = (fs[ot] - fs[a]).abs(); o = torch.argsort(dd)
            if dd[o[-1]] <= dd[o[0]]: continue
            rt.append(F.relu(torch.tensor(margin, device=dev) -
                      ((fz[a]*fz[ot[o[0]]]).sum() - (fz[a]*fz[ot[o[-1]]]).sum()) / temp))
    l_rk = torch.stack(rt).mean() if rt else torch.zeros((), device=dev)
    return lambda_multipos * l_mp + lambda_rank * l_rk

K_SERT  = lambda w: (lambda d, u: ((d > 0) & (d <= w) & u).float())
K_USTEL = lambda t: (lambda d, u: torch.exp(-d.float() / t) * u.float())

def egit_sacl(cekirdek, hn_disla, seed, epochs=75, lambda_seq=0.1, patience=4, eval_every=5):
    t0 = time.time(); torch.manual_seed(seed)
    model = SelfInferringHead().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ci  = itertools.cycle(chain_loader_hn)
    best = copy.deepcopy(model.state_dict()); best_r1 = -1.0; best_ep = 0; kotu = 0; ep_run = 0
    for ep in range(epochs):
        ep_run = ep + 1; model.train()
        for b in train_loader:
            img = b["img"].to(device); txt = b["txt"].to(device)
            hn  = b["hn"].to(device);  has = b["has_hn"].to(device); mi = b["mode_idx"].to(device)
            opt.zero_grad()
            mvi, ml = model.infer_mode(img, img); mvt, _ = model.infer_mode(txt, txt)
            zi = model.img_proj(img, mvi); zt = model.txt_proj(txt, mvt)
            zhn = model.project_image_with_vec(hn, mvi) if has.sum() > 0 else None
            loss = hn_info_nce(zi, zt, z_hn=zhn, has_hn=has)
            m = has.bool()
            if m.sum() > 0: loss = loss + MODE_AUX_WEIGHT * F.cross_entropy(ml[m], mi[m])
            cb = next(ci).to(device)
            loss = loss + lambda_seq * sacl_maskeli(
                project_chain_batch_self_infer(model, cb), cb, cekirdek, hn_disla=hn_disla)
            loss.backward(); opt.step()
        sch.step()
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            r = evaluate_self_paired(model, val_pairs)["R@1"]
            if r > best_r1:
                best_r1, best_ep, kotu = r, ep + 1, 0
                best = copy.deepcopy(model.state_dict())
            else:
                kotu += 1
                if kotu >= patience: break
    model.load_state_dict(best)
    return model, {"epochs_run": ep_run, "best_epoch": best_ep,
                   "best_val_r1": best_r1, "elapsed_sec": time.time() - t0}

KOLLAR_SACL = [
    ("A_hn_DISLANMIS", K_SERT(2),    True),   # asil soru: once bunlar kossun
    ("B_hn_DISLANMIS", K_USTEL(5.0), True),
    ("A_hn_pozitifte", K_SERT(2),    False),  # ayni kod yolundan kontrol
    ("B_hn_pozitifte", K_USTEL(5.0), False),
]

SONUC_SACL = pickle.load(open(CKPT_SACL, "rb")) if os.path.exists(CKPT_SACL) else {}
for ad, cek, disla in KOLLAR_SACL:
    SONUC_SACL.setdefault(ad, {})
    for sd in SEEDS_SACL:
        if sd in SONUC_SACL[ad]: continue
        print(f"\n=== {ad} | seed={sd} ===")
        m, meta = egit_sacl(cek, disla, sd)
        meta.update(evaluate_self_paired(m, test_pairs))
        SONUC_SACL[ad][sd] = meta
        print(f"  epoch={meta['epochs_run']} en_iyi={meta['best_epoch']} "
              f"sure={meta['elapsed_sec']/60:.1f}dk  R@1={meta['R@1']:.1f}%")
        pickle.dump(SONUC_SACL, open(CKPT_SACL, "wb")); del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"\n{'kol':<22}{'R@1 (%)':>18}{'MRR':>18}{'dk':>8}")
print("-" * 66)
for ad, _, _ in KOLLAR_SACL:
    d = SONUC_SACL.get(ad, {})
    if len(d) < len(SEEDS_SACL):
        print(f"{ad:<22}{f'({len(d)}/{len(SEEDS_SACL)} seed)':>18}"); continue
    r1 = np.array([d[s]["R@1"] for s in SEEDS_SACL]); mr = np.array([d[s]["MRR"] for s in SEEDS_SACL])
    print(f"{ad:<22}{f'{r1.mean():.2f} ± {r1.std(ddof=1):.2f}':>18}"
          f"{f'{mr.mean():.3f} ± {mr.std(ddof=1):.3f}':>18}"
          f"{np.mean([d[s]['elapsed_sec'] for s in SEEDS_SACL])/60:>8.1f}")
print("-" * 66)
try:
    _ref = MULTISEED_SUMMARY["D_leakfree"]
    print(f"Referans: zincir kaybi YOK (COSMOS ZoomInOut) = "
          f"{_ref['R@1_mean']:.2f} ± {_ref['R@1_std']:.2f}, {_ref['train_min_mean']:.1f} dk")
except Exception:
    print("Referans: zincir kaybi YOK (COSMOS ZoomInOut) = 52.35 ± 1.97, 0.8 dk")

try:
    from scipy import stats
    print()
    for kd, kp in [("A_hn_DISLANMIS", "A_hn_pozitifte"), ("B_hn_DISLANMIS", "B_hn_pozitifte")]:
        if len(SONUC_SACL.get(kd, {})) == len(SEEDS_SACL) and len(SONUC_SACL.get(kp, {})) == len(SEEDS_SACL):
            t, p = stats.ttest_rel([SONUC_SACL[kd][s]["R@1"] for s in SEEDS_SACL],
                                   [SONUC_SACL[kp][s]["R@1"] for s in SEEDS_SACL])
            print(f"  {kd} vs {kp} (eslesmis t-testi): t={t:+.3f}, p={p:.4f}")
except Exception as e:
    print("t-testi atlandi:", e)


---
## 27. Mod Etiketi Kontrolü

"Yalnız mod koşullama" kolu mimariyi ve yardımcı kaybı birlikte değiştiriyor;
ikisi hiç ayrılmadı. Bütün düzeyinde mod etiketi sabit olmasına rağmen kazanç
daha büyük çıktı. Bu hücre etiketin içeriğini bozup mimariyi sabit tutarak
hangisinin kazandırdığını belirler.

### ÇALIŞTIRMA SIRASI

**Run All YAPMA.**

1. Baştan başlayıp **"18. Leak-Free D / D + SACL-lite Eğitimi"** bölümünün
   sonuna kadar sırayla çalıştır. Yaklaşık 10 dakika, çoğu önbellekten gelir.
   Kesme noktası burası çünkü bu deneyin ihtiyaç duyduğu son tanım olan
   `evaluate_self_paired` orada tanımlanıyor.

2. **"18b. Vanilla-Random-Pairs" ile "25. Pencere Genişliği Deneyi" arasındaki
   her şeyi ATLA.** Hepsi önceki deneyler. Özellikle Bölüm 25 tamamlanmamış
   durumda ve çalıştırılırsa 80 dakika sürer.

3. Aşağıdaki hücreyi çalıştır. Yaklaşık 11 dakika.

**Kontrol:** Bölüm 9'da "Train zaten var, diskten yükleniyor" görmelisin.
"Sıfırdan üretiliyor" yazarsa DURDUR; veri kümesi değişir ve sonuçlar mevcut
tablolarla kıyaslanamaz hâle gelir.


In [ ]:
# HÜCRE MOD-KONTROL: mod etiketinin İÇERİĞİ mi, yoksa mimari mi kazandırıyor?
# ---------------------------------------------------------------------------
# SORU: "yalnız mod koşullama" kolu iki şeyi birden değiştiriyor -- mimariyi
# (ProjectionHead -> SelfInferringHead) ve mod yardimci kaybini. Bu ikisi hic
# ayrilmadi. Butun duzeyinde mod etiketi SABIT ("part") oldugu halde kazanc
# +12.05 cikti; bu, isi yapanin etiket icerigi degil mimari olabilecegini
# dusunduruyor.
#
# DENEY: mimari ve kayip AYNI kalir, yalnizca etiketin ICERIGI bozulur.
#   gercek      -> motorun verdigi renk/sekil/doku etiketi
#   karistirilmis -> ayni etiketler, ama baska orneklere atanmis
#   sabit       -> hepsi tek sinif (butun duzeyindeki dejenere durumun esi)
#
# OKUMA: karistirilmis ve sabit kollar gercek kolun belirgin altinda kalirsa
#        etiket icerigi belirleyicidir. Uc kol da ayni cikarsa kazanci ureten
#        sey mimaridir ve makaledeki iddia yeniden yazilmalidir.
import os, time, copy, pickle, random
import numpy as np, torch, torch.nn.functional as F
from torch.utils.data import DataLoader

CKPT_MK = f"{DRIVE_BASE}/ZoomInOut/v5i_results/mode_label_control.pkl"
SEEDS_MK = [42, 123, 777, 2024, 31415]

# ---------- etiket varyantlari ----------
_hn_idx = [i for i, p in enumerate(train_pairs) if p.get("hard_negative_embedding") is not None]
_modlar = [train_pairs[i].get("hard_negative_mode") for i in _hn_idx]
from collections import Counter
print(f"hard negative tasiyan cift: {len(_hn_idx)}/{len(train_pairs)} "
      f"(%{len(_hn_idx)/len(train_pairs)*100:.1f})")
print("gercek mod dagilimi:", dict(Counter(_modlar)))

def _varyant(tur):
    P = [dict(p) for p in train_pairs]
    if tur == "gercek":
        return P
    if tur == "karistirilmis":
        k = _modlar[:]; random.Random(0).shuffle(k)
        for i, m in zip(_hn_idx, k): P[i]["hard_negative_mode"] = m
    elif tur == "sabit":
        for i in _hn_idx: P[i]["hard_negative_mode"] = "color"
    return P

# ---------- egitim: MODE-ONLY, sizintisiz, HN itme YOK ----------
def egit_mode_only(loader, seed, epochs=75, patience=4, eval_every=5):
    t0 = time.time(); torch.manual_seed(seed)
    model = SelfInferringHead().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best = copy.deepcopy(model.state_dict()); best_r1 = -1.0; kotu = 0; ep_run = 0
    for ep in range(epochs):
        ep_run = ep + 1; model.train()
        for b in loader:
            img = b["img"].to(device); txt = b["txt"].to(device)
            has = b["has_hn"].to(device); mi = b["mode_idx"].to(device)
            opt.zero_grad()
            mvi, ml = model.infer_mode(img, img)
            mvt, _  = model.infer_mode(txt, txt)
            zi = model.img_proj(img, mvi); zt = model.txt_proj(txt, mvt)
            loss = plain_info_nce(zi, zt, z_hn=None, has_hn=has)
            m = has.bool()
            if m.sum() > 0:
                loss = loss + MODE_AUX_WEIGHT * F.cross_entropy(ml[m], mi[m])
            loss.backward(); opt.step()
        sch.step()
        if (ep + 1) % eval_every == 0 or ep + 1 == epochs:
            r = evaluate_self_paired(model, val_pairs)["R@1"]
            if r > best_r1: best_r1, kotu = r, 0; best = copy.deepcopy(model.state_dict())
            else:
                kotu += 1
                if kotu >= patience: break
    model.load_state_dict(best)
    return model, {"epochs_run": ep_run, "best_val_r1": best_r1, "elapsed_sec": time.time() - t0}

SONUC_MK = pickle.load(open(CKPT_MK, "rb")) if os.path.exists(CKPT_MK) else {}
for tur in ["gercek", "karistirilmis", "sabit"]:
    SONUC_MK.setdefault(tur, {})
    dl = DataLoader(PairDataset(_varyant(tur)), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    for sd in SEEDS_MK:
        if sd in SONUC_MK[tur]: continue
        m, meta = egit_mode_only(dl, sd)
        meta.update(evaluate_self_paired(m, test_pairs))
        SONUC_MK[tur][sd] = meta
        print(f"  {tur:<14} seed {sd:<6} {meta['elapsed_sec']/60:4.1f} dk  R@1={meta['R@1']:.1f}%")
        del m
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        pickle.dump(SONUC_MK, open(CKPT_MK, "wb"))

# ---------- ozet ----------
print(f"\n{'etiket':<16}{'R@1 (%)':>17}{'MRR':>17}")
print("-" * 50)
for tur in ["gercek", "karistirilmis", "sabit"]:
    d = SONUC_MK[tur]
    r1 = np.array([d[s]["R@1"] for s in SEEDS_MK]); mr = np.array([d[s]["MRR"] for s in SEEDS_MK])
    print(f"{tur:<16}{f'{r1.mean():.2f} ± {r1.std(ddof=1):.2f}':>17}"
          f"{f'{mr.mean():.3f} ± {mr.std(ddof=1):.3f}':>17}")
print("-" * 50)
print("Referans: vanilla, ZoomInOut ciftleri = 45.75 ± 1.40  (mod kosullama YOK)")

from scipy import stats
g = [SONUC_MK["gercek"][s]["R@1"] for s in SEEDS_MK]
for tur in ["karistirilmis", "sabit"]:
    x = [SONUC_MK[tur][s]["R@1"] for s in SEEDS_MK]
    t, p = stats.ttest_rel(g, x)
    print(f"  gercek vs {tur:<14} {np.mean(g)-np.mean(x):+6.2f} puan  t={t:+7.3f}  p={p:.4f}")
print("\nOKUMA: fark anlamsizsa kazanci uretenin etiket icerigi degil MIMARI")
print("       oldugu sonucu cikar ve makaledeki iddia yeniden yazilmalidir.")
